# Step 1: Chuẩn hóa dữ liệu đầu vào

## Mục tiêu của bước này

Bước đầu tiên nhằm chuẩn hóa hai tập dữ liệu chính được sử dụng cho giai đoạn Regression Analysis:

* `basket_df`: dữ liệu cấp độ giỏ hàng/hóa đơn, mỗi dòng đại diện cho một basket.
* `rules_df`: dữ liệu chứa 20 association rules đã được sinh ra từ bước Market Basket Analysis.

Do dữ liệu sản phẩm trong `basket_df["Items"]` và dữ liệu rule trong `rules_df["antecedents"]`, `rules_df["consequents"]` đang ở các định dạng khác nhau, cần chuyển chúng về cùng một dạng dữ liệu là `set`. Việc này giúp dễ dàng kiểm tra một basket có chứa các sản phẩm trong một rule hay không.

## Kết quả từ code

Sau khi chạy bước chuẩn hóa, dữ liệu thu được như sau:

* `basket_df` có kích thước ban đầu sau khi thêm cột mới là `(39516, 10)`.
* `rules_df` có kích thước sau khi thêm cột mới là `(20, 19)`.

Trong `basket_df`, hai cột mới đã được tạo:

* `Items_set`: tập hợp các sản phẩm trong từng basket.
* `ParsedBasketSize`: số lượng sản phẩm được đếm lại từ `Items_set`.

Trong `rules_df`, hai cột mới đã được tạo:

* `antecedents_set`: tập hợp sản phẩm ở vế trái của rule.
* `consequents_set`: tập hợp sản phẩm ở vế phải của rule.

Kết quả kiểm tra cho thấy số lượng sản phẩm trong `BasketSize` và số lượng sản phẩm được parse lại trong `ParsedBasketSize` hoàn toàn khớp nhau. Cụ thể:

```text
Number of BasketSize mismatches: 0
```

Điều này cho thấy quá trình chuyển đổi cột `Items` sang dạng `set` là chính xác.

## Kiểm tra missing values

Kết quả kiểm tra missing values cho thấy:

* Các cột quan trọng cho bước regression như `InvoiceNo`, `Items`, `BasketSize`, `ProductRevenue`, `TotalQuantity`, `AvgUnitPrice`, `Country`, `Items_set`, `ParsedBasketSize` không có giá trị thiếu.
* Cột `CustomerID` có 2,922 giá trị thiếu.

Tuy nhiên, trong giai đoạn regression hiện tại, `CustomerID` chưa phải là biến chính cần sử dụng. Vì vậy, missing values ở `CustomerID` chưa ảnh hưởng trực tiếp đến việc tạo biến `rule_applied` hay chạy mô hình hồi quy.

## Nhận xét

Kết quả của bước 1 cho thấy dữ liệu đã được chuẩn hóa đúng định dạng và có thể tiếp tục sử dụng cho các bước tiếp theo.

Đặc biệt, việc `BasketSize` khớp hoàn toàn với `ParsedBasketSize` chứng minh rằng quá trình parse danh sách sản phẩm trong từng basket không làm mất hoặc sai lệch dữ liệu sản phẩm.

Do đó, bước 1 được xem là hợp lệ và không cần chỉnh sửa thêm trước khi chuyển sang bước chọn rule và tạo biến kiểm tra rule.


In [8]:
# Import libraries for data processing
import pandas as pd
import numpy as np

# Import libraries for parsing item/rule strings
import ast
import re

# Import libraries for regression analysis
import statsmodels.api as sm
import statsmodels.formula.api as smf

# Import libraries for visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: "%.4f" % x)

# Define the 5 regression model formulas
model_formulas = {
    "Model 1 - Baseline": "AOV ~ any_rule_success",
    "Model 2 - Basket Size Control": "AOV ~ any_rule_success + basket_size",
    "Model 3 - Price Control": "AOV ~ any_rule_success + basket_size + avg_unit_price",
    "Model 4 - Country Control": "AOV ~ any_rule_success + basket_size + avg_unit_price + C(Country)",
    "Model 5 - Log AOV Robustness": "log_AOV ~ any_rule_success + basket_size + avg_unit_price + C(Country)"
}

print("Libraries imported successfully.")
print("Regression models defined successfully.")

Libraries imported successfully.
Regression models defined successfully.


In [9]:
# =========================
# STEP 1: STANDARDIZE DATA
# =========================

import pandas as pd
import numpy as np
import ast

# Copy data to avoid modifying the original dataframes
baskets = basket_df.copy()
rules = rules_df.copy()


# -------------------------
# Function to parse item collections
# -------------------------
def parse_item_collection(x):
    """
    Convert different item formats into a Python set of strings.

    Supported formats:
    - list: ['21232', '21523']
    - string list: "['21232', '21523']"
    - frozenset: frozenset({'22748', '22745'})
    - frozenset string: "frozenset({'22748', '22745'})"
    - set string: "{'22748', '22745'}"
    """

    # If already list/set/frozenset/tuple
    if isinstance(x, (list, set, frozenset, tuple)):
        return set(str(i).strip() for i in x)

    # If missing
    if pd.isna(x):
        return set()

    # Convert to string
    s = str(x).strip()

    # Handle frozenset string
    if s.startswith("frozenset(") and s.endswith(")"):
        s = s[len("frozenset("):-1]

    # Try safe Python parsing
    try:
        parsed = ast.literal_eval(s)

        if isinstance(parsed, (list, set, frozenset, tuple)):
            return set(str(i).strip() for i in parsed)
        else:
            return {str(parsed).strip()}

    except:
        pass

    # Fallback manual parsing
    s = s.replace("{", "").replace("}", "")
    s = s.replace("[", "").replace("]", "")
    s = s.replace("'", "").replace('"', "")

    if s == "":
        return set()

    return set(i.strip() for i in s.split(",") if i.strip() != "")


# -------------------------
# Standardize basket items
# -------------------------
baskets["Items_set"] = baskets["Items"].apply(parse_item_collection)

# Count parsed basket size
baskets["ParsedBasketSize"] = baskets["Items_set"].apply(len)


# -------------------------
# Standardize association rules
# -------------------------
rules["antecedents_set"] = rules["antecedents"].apply(parse_item_collection)
rules["consequents_set"] = rules["consequents"].apply(parse_item_collection)


# -------------------------
# Basic checks
# -------------------------
print("basket_df shape:", baskets.shape)
print("rules_df shape:", rules.shape)

print("\n===== basket_df columns =====")
print(baskets.columns.tolist())

print("\n===== rules_df columns =====")
print(rules.columns.tolist())


# -------------------------
# Missing values check
# -------------------------
print("\n===== Missing values in basket_df =====")
print(baskets.isna().sum())

print("\n===== Missing values in rules_df =====")
print(rules.isna().sum())


# -------------------------
# Check if BasketSize matches parsed item count
# -------------------------
print("\n===== Check BasketSize vs ParsedBasketSize =====")
display(baskets[["InvoiceNo", "BasketSize", "ParsedBasketSize"]].head(10))

mismatch_count = (baskets["BasketSize"] != baskets["ParsedBasketSize"]).sum()
print("\nNumber of BasketSize mismatches:", mismatch_count)


# -------------------------
# Preview parsed basket data
# -------------------------
print("\n===== Sample parsed basket =====")
display(
    baskets[
        ["InvoiceNo", "Items", "Items_set", "BasketSize", "ParsedBasketSize", "ProductRevenue"]
    ].head(3)
)


# -------------------------
# Preview parsed rule data
# -------------------------
print("\n===== Sample parsed rules =====")
display(
    rules[
        ["rule", "antecedents_set", "consequents_set", "support", "confidence", "lift"]
    ].head()
)

basket_df shape: (39516, 10)
rules_df shape: (20, 19)

===== basket_df columns =====
['InvoiceNo', 'Items', 'BasketSize', 'ProductRevenue', 'TotalQuantity', 'AvgUnitPrice', 'CustomerID', 'Country', 'Items_set', 'ParsedBasketSize']

===== rules_df columns =====
['antecedents', 'consequents', 'antecedent support', 'consequent support', 'support', 'confidence', 'lift', 'representativity', 'leverage', 'conviction', 'zhangs_metric', 'jaccard', 'certainty', 'kulczynski', 'antecedents_str', 'consequents_str', 'rule', 'antecedents_set', 'consequents_set']

===== Missing values in basket_df =====
InvoiceNo              0
Items                  0
BasketSize             0
ProductRevenue         0
TotalQuantity          0
AvgUnitPrice           0
CustomerID          2922
Country                0
Items_set              0
ParsedBasketSize       0
dtype: int64

===== Missing values in rules_df =====
antecedents           0
consequents           0
antecedent support    0
consequent support    0
suppor

,InvoiceNo,BasketSize,ParsedBasketSize
0,489434,8,8
1,489435,4,4
2,489436,19,19
3,489437,23,23
4,489438,17,17
5,489439,18,18
6,489440,2,2
7,489441,4,4
8,489442,23,23
9,489443,7,7



Number of BasketSize mismatches: 0

===== Sample parsed basket =====


,InvoiceNo,Items,Items_set,BasketSize,ParsedBasketSize,ProductRevenue
0,489434,"['21232', '21523', '21871', '22041', '22064', ...","{21871, 22041, 22064, 85048, 79323W, 79323P, 2...",8,8,505.3000
1,489435,"['22195', '22349', '22350', '22353']","{22353, 22195, 22350, 22349}",4,4,145.8000
2,489436,"['21181', '21333', '21754', '21755', '21756', ...","{21755, 84879, 22295, 21756, 22142, 21333, 221...",19,19,630.3300



===== Sample parsed rules =====


,rule,antecedents_set,consequents_set,support,confidence,lift
0,"22748, 22745 → 22746","{22745, 22748}",{22746},0.0104,0.7411,52.6464
1,"22746 → 22748, 22745",{22746},"{22745, 22748}",0.0104,0.7411,52.6464
2,"22748, 22746 → 22745","{22748, 22746}",{22745},0.0104,0.8701,50.6062
3,"22745 → 22748, 22746",{22745},"{22748, 22746}",0.0104,0.6068,50.6062
4,22300 → 22301,{22300},{22301},0.0100,0.7627,48.7823


# Step 2: Chọn một association rule và kiểm tra mức độ xuất hiện trong basket data

## Mục tiêu của bước này

Sau khi chuẩn hóa dữ liệu, bước tiếp theo là chọn một association rule cụ thể để kiểm tra xem rule đó xuất hiện như thế nào trong dữ liệu basket.

Rule được chọn là rule đầu tiên trong `rules_df`:

```text
22748, 22745 → 22746
```

Trong rule này:

* Vế trái, hay `antecedents`, gồm hai sản phẩm: `22745` và `22748`.
* Vế phải, hay `consequents`, gồm một sản phẩm: `22746`.

Mục tiêu của bước này là kiểm tra:

1. Có bao nhiêu basket chứa vế trái của rule.
2. Có bao nhiêu basket chứa vế phải của rule.
3. Có bao nhiêu basket chứa đầy đủ cả vế trái và vế phải.
4. Có bao nhiêu basket là ứng viên để gợi ý cross-sell.

## Chỉ số association rule ban đầu

Rule được chọn có các chỉ số từ bước Market Basket Analysis như sau:

| Chỉ số     | Giá trị |
| ---------- | ------: |
| Support    |  0.0104 |
| Confidence |  0.7411 |
| Lift       | 52.6464 |

Các chỉ số này cho thấy rule có mối liên hệ mạnh giữa nhóm sản phẩm ở vế trái và sản phẩm ở vế phải.

Cụ thể, confidence bằng khoảng 0.7411 nghĩa là trong các basket có chứa `22745` và `22748`, khoảng 74.11% basket cũng chứa thêm `22746`.

Lift bằng 52.6464 là một giá trị rất cao, cho thấy ba sản phẩm này có xu hướng xuất hiện cùng nhau mạnh hơn rất nhiều so với trường hợp ngẫu nhiên.

## Kết quả kiểm tra trên basket data

Tổng số basket trong dữ liệu là:

```text
39,516 baskets
```

Kết quả kiểm tra rule trong dữ liệu basket như sau:

| Điều kiện kiểm tra                 | Số lượng basket |  Tỷ lệ |
| ---------------------------------- | --------------: | -----: |
| Basket chứa antecedents            |             557 | 0.0141 |
| Basket chứa consequents            |             564 | 0.0143 |
| Basket chứa full rule              |             409 | 0.0104 |
| Basket là recommendation candidate |             148 | 0.0037 |

Trong đó:

* `antecedent_present` nghĩa là basket có chứa cả hai sản phẩm `22745` và `22748`.
* `consequent_present` nghĩa là basket có chứa sản phẩm `22746`.
* `full_rule_present` nghĩa là basket có chứa cả `22745`, `22748` và `22746`.
* `recommendation_candidate` nghĩa là basket có chứa `22745` và `22748` nhưng chưa chứa `22746`.

## Diễn giải kết quả

Có 557 basket chứa vế trái của rule, tức là có 557 basket mà khách hàng đã mua cùng lúc hai sản phẩm `22745` và `22748`.

Trong số đó, có 409 basket cũng chứa sản phẩm `22746`. Điều này phù hợp với confidence cao của rule, vì phần lớn basket có vế trái cũng chứa thêm vế phải.

Ngoài ra, có 148 basket được xác định là recommendation candidates. Đây là các basket đã chứa `22745` và `22748`, nhưng chưa chứa `22746`.

Về mặt cross-selling, nhóm 148 basket này rất quan trọng vì đây là nhóm có thể được dùng để mô phỏng tình huống gợi ý thêm sản phẩm `22746`.

Nói cách khác, nếu hệ thống recommendation phát hiện khách hàng đang mua `22745` và `22748`, hệ thống có thể đề xuất thêm `22746` như một sản phẩm cross-sell.

## Nhận xét quan trọng

Rule này đủ điều kiện để tiếp tục phân tích vì:

* Rule xuất hiện thực tế trong dữ liệu basket.
* Số lượng basket chứa full rule là 409, không quá nhỏ.
* Số lượng recommendation candidates là 148, cho thấy có nhóm basket tiềm năng để mô phỏng cross-sell.

Tuy nhiên, khi xem sample basket, có thể thấy một số basket chứa rule có `BasketSize` rất lớn, ví dụ 30, 100 hoặc 110 sản phẩm.

Điều này cần được chú ý vì AOV cao có thể không đến từ bản thân rule, mà có thể đến từ việc basket đó vốn đã có nhiều sản phẩm. Nếu không kiểm soát yếu tố này, mô hình hồi quy có thể đưa ra kết luận sai lệch.

Vì vậy, trước khi chạy regression, cần thực hiện bước kiểm tra mô tả giữa hai nhóm:

* Nhóm `rule_applied = 1`: basket có chứa đầy đủ rule.
* Nhóm `rule_applied = 0`: basket không chứa đầy đủ rule.

Bước tiếp theo cần kiểm tra sự khác biệt về:

* AOV
* BasketSize
* TotalQuantity
* AvgUnitPrice
* Outliers

Việc này giúp xác định xem có cần xử lý outlier hoặc thêm biến kiểm soát trước khi chạy mô hình hồi quy hay không.

## Kết luận bước 2

Bước 2 cho thấy rule `22748, 22745 → 22746` là một rule hợp lệ để tiếp tục phân tích regression.

Tuy nhiên, do rule có thể xuất hiện trong các basket lớn, chưa nên chạy regression ngay. Cần tiếp tục kiểm tra thống kê mô tả và phân phối dữ liệu ở bước tiếp theo để tránh kết luận sai về tác động của rule lên AOV.


In [10]:
# ============================================
# STEP 2: SELECT ONE RULE AND CHECK ITS COVERAGE
# ============================================

# Select rule index
# Start with the top rule from rules_df
RULE_INDEX = 0

# Get selected rule
selected_rule = rules.loc[RULE_INDEX]

antecedents = selected_rule["antecedents_set"]
consequents = selected_rule["consequents_set"]
all_rule_items = antecedents.union(consequents)

print("===== Selected Rule =====")
print("Rule index:", RULE_INDEX)
print("Rule:", selected_rule["rule"])
print("Antecedents:", antecedents)
print("Consequents:", consequents)

print("\n===== Rule Metrics from Association Rule Mining =====")
print("Support:", selected_rule["support"])
print("Confidence:", selected_rule["confidence"])
print("Lift:", selected_rule["lift"])


# -------------------------
# Create temporary checking variables
# -------------------------
temp_check = baskets.copy()

temp_check["antecedent_present"] = temp_check["Items_set"].apply(
    lambda items: antecedents.issubset(items)
)

temp_check["consequent_present"] = temp_check["Items_set"].apply(
    lambda items: consequents.issubset(items)
)

temp_check["full_rule_present"] = temp_check["Items_set"].apply(
    lambda items: all_rule_items.issubset(items)
)

temp_check["recommendation_candidate"] = (
    temp_check["antecedent_present"] & 
    ~temp_check["consequent_present"]
)


# -------------------------
# Count rule coverage
# -------------------------
total_baskets = len(temp_check)

antecedent_count = temp_check["antecedent_present"].sum()
consequent_count = temp_check["consequent_present"].sum()
full_rule_count = temp_check["full_rule_present"].sum()
candidate_count = temp_check["recommendation_candidate"].sum()

print("\n===== Rule Coverage in Basket Data =====")
print("Total baskets:", total_baskets)

print("\nBaskets containing antecedents:", antecedent_count)
print("Antecedent coverage rate:", round(antecedent_count / total_baskets, 4))

print("\nBaskets containing consequents:", consequent_count)
print("Consequent coverage rate:", round(consequent_count / total_baskets, 4))

print("\nBaskets containing full rule:", full_rule_count)
print("Full rule coverage rate:", round(full_rule_count / total_baskets, 4))

print("\nPotential recommendation candidates:")
print("Baskets with antecedents but without consequents:", candidate_count)
print("Candidate rate:", round(candidate_count / total_baskets, 4))


# -------------------------
# Display sample baskets
# -------------------------
print("\n===== Sample baskets containing full rule =====")
display(
    temp_check[temp_check["full_rule_present"] == True][
        ["InvoiceNo", "Items_set", "BasketSize", "ProductRevenue", 
         "antecedent_present", "consequent_present", "full_rule_present"]
    ].head()
)

print("\n===== Sample recommendation candidates =====")
display(
    temp_check[temp_check["recommendation_candidate"] == True][
        ["InvoiceNo", "Items_set", "BasketSize", "ProductRevenue", 
         "antecedent_present", "consequent_present", "recommendation_candidate"]
    ].head()
)

===== Selected Rule =====
Rule index: 0
Rule: 22748, 22745 → 22746
Antecedents: {'22745', '22748'}
Consequents: {'22746'}

===== Rule Metrics from Association Rule Mining =====
Support: 0.0104326053693142
Confidence: 0.7411067193675889
Lift: 52.64640519301973

===== Rule Coverage in Basket Data =====
Total baskets: 39516

Baskets containing antecedents: 557
Antecedent coverage rate: 0.0141

Baskets containing consequents: 564
Consequent coverage rate: 0.0143

Baskets containing full rule: 409
Full rule coverage rate: 0.0104

Potential recommendation candidates:
Baskets with antecedents but without consequents: 148
Candidate rate: 0.0037

===== Sample baskets containing full rule =====


,InvoiceNo,Items_set,BasketSize,ProductRevenue,antecedent_present,consequent_present,full_rule_present
14314,523929,"{22748, 22468, 22944, 22940, 22907, 22746, 228...",30,8880.4200,True,True,True
14344,523960,"{22733, 20972, 22562, 85049E, 22566, 21412, 22...",100,595.8900,True,True,True
14345,523961,"{22910, 22748, 21705, 22635, 21056, 22622, 226...",16,295.7300,True,True,True
14348,523964,"{22748, 22627, 22944, 21491, 22727, 22746, 226...",32,717.3000,True,True,True
14356,523973,"{22748, 22562, 22746, 22898, 22666, 22642, 224...",20,422.4500,True,True,True



===== Sample recommendation candidates =====


,InvoiceNo,Items_set,BasketSize,ProductRevenue,antecedent_present,consequent_present,recommendation_candidate
14298,523891,"{22745, 22748}",2,4.2000,True,False,True
14325,523940,"{22905, 22748, 22904, 22903, 22745}",5,96.0000,True,False,True
14342,523958,"{48138, 22748, 22412, 22726, 22804, 47566, 727...",11,416.0800,True,False,True
14438,524125,"{22554, 20972, 22733, 22662, 22909, 15056BL, 2...",110,2245.9300,True,False,True
14501,524264,"{22505, 22748, 21222, 21034, 21358, 21591, 214...",57,366.1600,True,False,True


# Step 3: Tạo regression dataset và kiểm tra thống kê mô tả

## Mục tiêu của bước này

Bước 3 nhằm tạo bộ dữ liệu ở cấp độ basket để chuẩn bị cho giai đoạn hồi quy. Mục tiêu chính là xây dựng biến phụ thuộc, biến độc lập chính và các biến kiểm soát cần thiết cho mô hình regression.

Trong bước này, rule được chọn để phân tích là:

```text
22748, 22745 → 22746
```

Với rule này, biến `rule_applied` được tạo như sau:

* `rule_applied = 1` nếu basket chứa đầy đủ cả vế trái và vế phải của rule.
* `rule_applied = 0` nếu basket không chứa đầy đủ rule.

Biến phụ thuộc của mô hình là `AOV`, được lấy từ cột `ProductRevenue`. Trong dữ liệu này, mỗi dòng đại diện cho một basket, vì vậy `ProductRevenue` được xem là tổng giá trị của basket.

Các biến kiểm soát hiện có trong dữ liệu gồm:

* `BasketSize`: số lượng sản phẩm khác nhau trong basket.
* `TotalQuantity`: tổng số lượng sản phẩm được mua trong basket.
* `AvgUnitPrice`: đơn giá trung bình của sản phẩm trong basket.
* `Country`: quốc gia của giao dịch.

Dữ liệu không có biến `promo`, do đó chưa thể kiểm soát trực tiếp yếu tố khuyến mãi trong mô hình.

## Kết quả tạo regression dataset

Sau khi tạo dữ liệu cho regression, `model_df` có kích thước:

```text
(39516, 12)
```

Các cột chính trong `model_df` gồm:

```text
InvoiceNo
AOV
rule_applied
antecedent_present
consequent_present
recommendation_candidate
BasketSize
TotalQuantity
AvgUnitPrice
CustomerID
Country
Items_set
```

Kết quả kiểm tra missing values cho thấy các biến quan trọng cho mô hình như `AOV`, `rule_applied`, `BasketSize`, `TotalQuantity`, `AvgUnitPrice` và `Country` không có giá trị thiếu. Cột `CustomerID` có 2,922 giá trị thiếu, tuy nhiên cột này chưa được sử dụng trong mô hình hồi quy hiện tại nên không ảnh hưởng trực tiếp đến bước phân tích này.

## Phân phối biến rule_applied

Kết quả phân phối của biến `rule_applied` như sau:

| rule_applied | Số lượng basket |
| -----------: | --------------: |
|            0 |          39,107 |
|            1 |             409 |

Tỷ lệ basket có `rule_applied = 1` là khoảng 0.0104, tương đương 1.04% tổng số basket.

Kết quả này khớp với phần kiểm tra rule ở bước trước, cho thấy biến `rule_applied` đã được tạo đúng.

## Phân phối recommendation candidates

Biến `recommendation_candidate` xác định các basket có chứa vế trái của rule nhưng chưa chứa vế phải. Đây là nhóm basket có thể được dùng để mô phỏng tình huống gợi ý cross-sell.

Kết quả cho thấy:

| recommendation_candidate | Số lượng basket |
| -----------------------: | --------------: |
|                        0 |          39,368 |
|                        1 |             148 |

Như vậy, có 148 basket có thể được xem là ứng viên để gợi ý thêm sản phẩm `22746` khi khách hàng đã mua `22745` và `22748`.

## Thống kê mô tả tổng thể

Thống kê mô tả cho các biến numeric chính cho thấy:

| Biến          |   Mean | Median |        Max |
| ------------- | -----: | -----: | ---------: |
| AOV           | 497.11 | 301.23 | 168,469.60 |
| BasketSize    |  25.11 |     15 |      1,108 |
| TotalQuantity | 283.13 |    148 |     87,167 |
| AvgUnitPrice  |   3.66 |   2.84 |   1,157.15 |

Các giá trị lớn nhất của `AOV`, `BasketSize`, `TotalQuantity` và `AvgUnitPrice` đều cao hơn rất nhiều so với median. Điều này cho thấy dữ liệu có hiện tượng outlier mạnh.

## So sánh giữa nhóm có rule và không có rule

Khi so sánh hai nhóm `rule_applied = 0` và `rule_applied = 1`, kết quả cho thấy:

| Nhóm             | Mean AOV | Median AOV | Mean BasketSize | Median BasketSize |
| ---------------- | -------: | ---------: | --------------: | ----------------: |
| rule_applied = 0 |   489.08 |     300.60 |           24.49 |                15 |
| rule_applied = 1 | 1,265.11 |     490.87 |           84.10 |                46 |

Nhóm basket có `rule_applied = 1` có AOV trung bình và AOV trung vị cao hơn nhóm còn lại. Tuy nhiên, nhóm này cũng có `BasketSize` lớn hơn rất nhiều.

Điều này cho thấy AOV cao hơn ở nhóm có rule chưa chắc đến từ bản thân rule. Một khả năng hợp lý là các basket có rule thường là những basket lớn hơn, có nhiều sản phẩm hơn, nên tổng giá trị đơn hàng cũng cao hơn.

Vì vậy, nếu chỉ chạy mô hình đơn giản `AOV ~ rule_applied`, kết quả có thể bị sai lệch. Cần thêm các biến kiểm soát như `BasketSize`, `TotalQuantity` và `AvgUnitPrice` để tách bớt ảnh hưởng của quy mô giỏ hàng.

## Kiểm tra outlier

Kết quả kiểm tra quantile cho thấy dữ liệu có outlier rất mạnh. Ví dụ:

| Biến          | Median | 99% quantile |        Max |
| ------------- | -----: | -----------: | ---------: |
| AOV           | 301.23 |     4,103.92 | 168,469.60 |
| BasketSize    |     15 |       196.85 |      1,108 |
| TotalQuantity |    148 |        2,224 |     87,167 |
| AvgUnitPrice  |   2.84 |        12.75 |   1,157.15 |

Khoảng cách rất lớn giữa giá trị 99% và giá trị lớn nhất cho thấy một số basket cực đoan có thể ảnh hưởng mạnh đến kết quả hồi quy.

Ngoài ra, một số outlier cũng nằm trong nhóm `rule_applied = 1`. Ví dụ, có basket có `rule_applied = 1` với AOV hơn 49,000 hoặc BasketSize lên tới 1,108 sản phẩm. Những quan sát này có thể làm hệ số của `rule_applied` bị phóng đại nếu chạy hồi quy trực tiếp trên dữ liệu raw.

## Nhận xét

Bước 3 cho thấy dữ liệu đã sẵn sàng về mặt cấu trúc để chạy regression. Tuy nhiên, dữ liệu có hai vấn đề quan trọng cần xử lý trước khi chạy mô hình:

1. Nhóm `rule_applied = 1` có BasketSize lớn hơn rất nhiều so với nhóm `rule_applied = 0`.
2. Dữ liệu có outlier mạnh ở các biến `AOV`, `BasketSize`, `TotalQuantity` và `AvgUnitPrice`.

Do đó, chưa nên chạy regression ngay trên dữ liệu gốc. Cần thực hiện bước xử lý outlier và tạo biến log cho AOV để mô hình ổn định hơn.

## Kết luận bước 3

Regression dataset đã được tạo thành công và biến `rule_applied` được xác định đúng. Tuy nhiên, do dữ liệu có outlier mạnh và sự khác biệt lớn về BasketSize giữa hai nhóm, bước tiếp theo cần xử lý outlier và tạo biến `log_AOV` trước khi chạy mô hình hồi quy.


In [11]:
# ============================================================
# STEP 3: CREATE REGRESSION DATASET AND DESCRIPTIVE DIAGNOSTICS
# ============================================================

# We continue using the same selected rule from Step 2
RULE_INDEX = 0

selected_rule = rules.loc[RULE_INDEX]

antecedents = selected_rule["antecedents_set"]
consequents = selected_rule["consequents_set"]
all_rule_items = antecedents.union(consequents)

print("===== Selected Rule for Regression Dataset =====")
print("Rule index:", RULE_INDEX)
print("Rule:", selected_rule["rule"])
print("Antecedents:", antecedents)
print("Consequents:", consequents)


# -------------------------
# Create model dataframe
# -------------------------
model_df = baskets.copy()

# Check rule-related conditions
model_df["antecedent_present"] = model_df["Items_set"].apply(
    lambda items: antecedents.issubset(items)
)

model_df["consequent_present"] = model_df["Items_set"].apply(
    lambda items: consequents.issubset(items)
)

model_df["rule_applied"] = model_df["Items_set"].apply(
    lambda items: all_rule_items.issubset(items)
).astype(int)

model_df["recommendation_candidate"] = (
    model_df["antecedent_present"] & 
    ~model_df["consequent_present"]
).astype(int)

# Define AOV
model_df["AOV"] = model_df["ProductRevenue"]


# -------------------------
# Keep only regression-related columns
# -------------------------
model_df = model_df[
    [
        "InvoiceNo",
        "AOV",
        "rule_applied",
        "antecedent_present",
        "consequent_present",
        "recommendation_candidate",
        "BasketSize",
        "TotalQuantity",
        "AvgUnitPrice",
        "CustomerID",
        "Country",
        "Items_set"
    ]
].copy()


# -------------------------
# Basic structure check
# -------------------------
print("\n===== model_df Shape =====")
print(model_df.shape)

print("\n===== model_df Columns =====")
print(model_df.columns.tolist())

print("\n===== Missing Values =====")
print(model_df.isna().sum())


# -------------------------
# Check rule_applied distribution
# -------------------------
print("\n===== rule_applied Distribution =====")
rule_counts = model_df["rule_applied"].value_counts().sort_index()
display(rule_counts.to_frame("count"))

rule_rate = model_df["rule_applied"].mean()
print("Rule-applied rate:", round(rule_rate, 4))


# -------------------------
# Check recommendation candidate distribution
# -------------------------
print("\n===== recommendation_candidate Distribution =====")
candidate_counts = model_df["recommendation_candidate"].value_counts().sort_index()
display(candidate_counts.to_frame("count"))

candidate_rate = model_df["recommendation_candidate"].mean()
print("Recommendation candidate rate:", round(candidate_rate, 4))


# -------------------------
# Overall descriptive statistics
# -------------------------
print("\n===== Overall Numeric Descriptive Statistics =====")
display(
    model_df[
        ["AOV", "BasketSize", "TotalQuantity", "AvgUnitPrice"]
    ].describe().round(4)
)


# -------------------------
# Group comparison: rule_applied = 0 vs 1
# -------------------------
print("\n===== Group Comparison by rule_applied =====")
group_summary = model_df.groupby("rule_applied").agg(
    baskets=("InvoiceNo", "count"),
    mean_AOV=("AOV", "mean"),
    median_AOV=("AOV", "median"),
    mean_BasketSize=("BasketSize", "mean"),
    median_BasketSize=("BasketSize", "median"),
    mean_TotalQuantity=("TotalQuantity", "mean"),
    median_TotalQuantity=("TotalQuantity", "median"),
    mean_AvgUnitPrice=("AvgUnitPrice", "mean"),
    median_AvgUnitPrice=("AvgUnitPrice", "median")
).round(4)

display(group_summary)


# -------------------------
# Quantile check for outliers
# -------------------------
print("\n===== Quantile Check for Possible Outliers =====")
quantile_table = model_df[
    ["AOV", "BasketSize", "TotalQuantity", "AvgUnitPrice"]
].quantile([0.50, 0.75, 0.90, 0.95, 0.99, 0.995, 1.00]).round(4)

display(quantile_table)


# -------------------------
# Check top extreme AOV baskets
# -------------------------
print("\n===== Top 10 Highest AOV Baskets =====")
display(
    model_df.sort_values("AOV", ascending=False)[
        ["InvoiceNo", "AOV", "rule_applied", "BasketSize", "TotalQuantity", "AvgUnitPrice", "Country"]
    ].head(10)
)


# -------------------------
# Check top extreme BasketSize baskets
# -------------------------
print("\n===== Top 10 Largest BasketSize Baskets =====")
display(
    model_df.sort_values("BasketSize", ascending=False)[
        ["InvoiceNo", "AOV", "rule_applied", "BasketSize", "TotalQuantity", "AvgUnitPrice", "Country"]
    ].head(10)
)

===== Selected Rule for Regression Dataset =====
Rule index: 0
Rule: 22748, 22745 → 22746
Antecedents: {'22745', '22748'}
Consequents: {'22746'}

===== model_df Shape =====
(39516, 12)

===== model_df Columns =====
['InvoiceNo', 'AOV', 'rule_applied', 'antecedent_present', 'consequent_present', 'recommendation_candidate', 'BasketSize', 'TotalQuantity', 'AvgUnitPrice', 'CustomerID', 'Country', 'Items_set']

===== Missing Values =====
InvoiceNo                      0
AOV                            0
rule_applied                   0
antecedent_present             0
consequent_present             0
recommendation_candidate       0
BasketSize                     0
TotalQuantity                  0
AvgUnitPrice                   0
CustomerID                  2922
Country                        0
Items_set                      0
dtype: int64

===== rule_applied Distribution =====


,count
rule_applied,
0,39107
1,409


Rule-applied rate: 0.0104

===== recommendation_candidate Distribution =====


,count
recommendation_candidate,
0,39368
1,148


Recommendation candidate rate: 0.0037

===== Overall Numeric Descriptive Statistics =====


...

,AOV,BasketSize,TotalQuantity,AvgUnitPrice
0.5000,301.2300,15.0000,148.0000,2.8388
0.7500,484.4000,28.0000,289.0000,4.0000
0.9000,905.5600,51.0000,523.0000,5.6000
0.9500,1433.8575,73.0000,794.0000,6.9500
0.9900,4103.9240,196.8500,2224.0000,12.7500
0.9950,5743.1000,284.0000,3566.5500,19.1657
1.0000,168469.6000,1108.0000,87167.0000,1157.1500



===== Top 10 Highest AOV Baskets =====


,InvoiceNo,AOV,rule_applied,BasketSize,TotalQuantity,AvgUnitPrice,Country
39480,581483,168469.6000,0,1,80995,2.0800,United Kingdom
21850,541431,77183.6000,0,1,74215,1.0400,United Kingdom
36497,574941,52940.9400,0,101,14149,4.9395,United Kingdom
37146,576365,50653.9100,0,99,13956,4.7425,United Kingdom
18283,533027,49844.9900,1,111,13387,5.0963,United Kingdom
17623,531516,45332.9700,1,115,12410,5.6879,United Kingdom
1764,493819,44051.6000,0,94,25018,2.4276,EIRE
28375,556444,38970.0000,0,1,60,649.5000,United Kingdom
14462,524181,33167.8000,0,13,8172,4.3638,United Kingdom
33164,567423,31698.1600,0,12,12572,3.0642,United Kingdom



===== Top 10 Largest BasketSize Baskets =====


,InvoiceNo,AOV,rule_applied,BasketSize,TotalQuantity,AvgUnitPrice,Country
35936,573585,14838.8600,1,1108,5196,4.4925,United Kingdom
39357,581219,7150.0700,0,748,2150,4.2926,United Kingdom
39487,581492,6756.0600,0,730,2010,4.1759,United Kingdom
39129,580729,7848.7300,0,720,2455,4.5273,United Kingdom
29262,558475,7847.4300,0,703,4136,2.9736,United Kingdom
38676,579777,5880.6400,1,686,1762,4.2712,United Kingdom
39355,581217,6075.6200,0,674,1851,4.2500,United Kingdom
20264,537434,7272.4100,0,673,1868,4.7884,United Kingdom
39130,580730,6024.9700,1,660,1857,4.3822,United Kingdom
38932,580367,5933.1500,0,649,1871,3.9617,United Kingdom


# Step 4: Xử lý outlier và tạo biến log(AOV)

## Mục tiêu của bước này

Bước 4 nhằm xử lý các giá trị ngoại lai trước khi chạy mô hình hồi quy. Ở bước 3, thống kê mô tả cho thấy dữ liệu có outlier mạnh ở các biến như `AOV`, `BasketSize`, `TotalQuantity` và `AvgUnitPrice`.

Nếu chạy hồi quy trực tiếp trên dữ liệu gốc, một số basket cực đoan có thể ảnh hưởng mạnh đến hệ số hồi quy, đặc biệt là hệ số của biến `rule_applied`. Vì vậy, cần tạo một phiên bản dữ liệu sạch hơn trước khi chạy mô hình.

Ngoài ra, bước này cũng tạo thêm biến `log_AOV` để giảm ảnh hưởng của các giá trị AOV rất lớn và giúp diễn giải kết quả hồi quy theo tỷ lệ phần trăm.

## Kiểm tra điều kiện lấy log

Kết quả kiểm tra cho thấy:

```text
Minimum AOV: 0.19
Number of AOV <= 0: 0
```

Do tất cả giá trị AOV đều lớn hơn 0, có thể sử dụng log tự nhiên của AOV:

```text
log_AOV = log(AOV)
```

Việc log-transform AOV giúp giảm độ lệch phải của phân phối doanh thu và làm mô hình hồi quy ổn định hơn.

## Phương pháp xử lý outlier

Outlier được xác định bằng ngưỡng phân vị 99.5% cho bốn biến chính:

* `AOV`
* `BasketSize`
* `TotalQuantity`
* `AvgUnitPrice`

Một basket được xem là outlier nếu vượt quá ngưỡng 99.5% ở ít nhất một trong bốn biến trên.

Các ngưỡng 99.5% được xác định như sau:

| Biến          | Ngưỡng 99.5% |
| ------------- | -----------: |
| AOV           |      5743.10 |
| BasketSize    |          284 |
| TotalQuantity |      3566.55 |
| AvgUnitPrice  |      19.1657 |

## Kết quả loại outlier

Số lượng outlier theo từng biến như sau:

| Biến          | Số lượng outlier | Tỷ lệ |
| ------------- | ---------------: | ----: |
| AOV           |              198 | 0.50% |
| BasketSize    |              197 | 0.50% |
| TotalQuantity |              198 | 0.50% |
| AvgUnitPrice  |              198 | 0.50% |

Tổng số basket outlier duy nhất là:

```text
647 baskets
```

Tỷ lệ outlier tổng là:

```text
1.64%
```

Kích thước dữ liệu trước và sau khi loại outlier:

| Giai đoạn       | Số basket |
| --------------- | --------: |
| Trước cleaning  |    39,516 |
| Sau cleaning    |    38,869 |
| Số dòng bị loại |       647 |

Như vậy, bước xử lý outlier chỉ loại bỏ 1.64% dữ liệu, tức là không làm mất quá nhiều quan sát.

## Kiểm tra nhóm rule_applied sau cleaning

Trước khi loại outlier:

| rule_applied | Số basket |
| -----------: | --------: |
|            0 |    39,107 |
|            1 |       409 |

Sau khi loại outlier:

| rule_applied | Số basket |
| -----------: | --------: |
|            0 |    38,488 |
|            1 |       381 |

Nhóm `rule_applied = 1` giảm từ 409 xuống còn 381 basket, tức là chỉ mất 28 basket. Điều này cho thấy việc xử lý outlier không làm mất quá nhiều quan sát thuộc nhóm có rule. Do đó, dữ liệu sau cleaning vẫn phù hợp để chạy hồi quy.

## So sánh thống kê trước và sau cleaning

Trước khi cleaning, các biến có giá trị cực đại rất lớn. Ví dụ, AOV lớn nhất đạt 168,469.60 và BasketSize lớn nhất đạt 1,108 sản phẩm.

Sau khi cleaning, dữ liệu ổn định hơn:

| Biến          | Mean sau cleaning | Std sau cleaning | Max sau cleaning |
| ------------- | ----------------: | ---------------: | ---------------: |
| AOV           |            415.56 |           508.33 |          5737.32 |
| BasketSize    |             22.96 |            28.78 |              284 |
| TotalQuantity |            232.54 |           306.06 |             3564 |
| AvgUnitPrice  |              3.26 |             1.93 |            19.16 |

So với dữ liệu gốc, độ lệch chuẩn và giá trị cực đại của các biến đã giảm mạnh. Điều này giúp hạn chế ảnh hưởng của các basket cực đoan lên kết quả hồi quy.

## So sánh nhóm có rule và không có rule sau cleaning

Sau khi xử lý outlier, kết quả so sánh giữa hai nhóm như sau:

| Nhóm             | Mean AOV | Median AOV | Mean BasketSize | Median BasketSize |
| ---------------- | -------: | ---------: | --------------: | ----------------: |
| rule_applied = 0 |   412.66 |     298.85 |           22.62 |                15 |
| rule_applied = 1 |   707.98 |     459.50 |           56.96 |                44 |

Nhóm có `rule_applied = 1` vẫn có AOV trung bình và AOV trung vị cao hơn nhóm không có rule. Cụ thể, AOV trung bình của nhóm có rule cao hơn khoảng 295.32 đơn vị.

Tuy nhiên, nhóm có rule cũng có BasketSize cao hơn rõ rệt. Điều này cho thấy AOV cao hơn có thể không chỉ đến từ rule, mà còn đến từ việc các basket có rule thường là các basket lớn hơn.

Vì vậy, mô hình hồi quy cần kiểm soát thêm các biến như `BasketSize`, `TotalQuantity` và `AvgUnitPrice` để đánh giá chính xác hơn mối quan hệ giữa `rule_applied` và AOV.

## Nhận xét

Bước 4 đã xử lý được vấn đề outlier mạnh trong dữ liệu. Sau khi loại bỏ các basket cực đoan theo ngưỡng 99.5%, dữ liệu vẫn giữ lại phần lớn quan sát và vẫn còn đủ số lượng basket thuộc nhóm `rule_applied = 1`.

Dữ liệu sau cleaning phù hợp hơn để chạy hồi quy so với dữ liệu gốc. Tuy nhiên, kết quả thống kê mô tả vẫn cho thấy nhóm có rule có BasketSize lớn hơn đáng kể so với nhóm không có rule. Vì vậy, cần tiếp tục chạy mô hình hồi quy theo từng cấp độ: bắt đầu với mô hình baseline, sau đó thêm các biến kiểm soát.

## Kết luận bước 4

Dữ liệu đã được xử lý outlier và tạo biến `log_AOV` thành công. Bộ dữ liệu `model_reg_clean` có thể được sử dụng cho bước hồi quy tiếp theo.

Bước tiếp theo là chạy mô hình hồi quy baseline để kiểm tra mối quan hệ thô giữa `rule_applied` và AOV trước khi thêm các biến kiểm soát.


In [12]:
# ===================================================
# STEP 4: HANDLE OUTLIERS AND CREATE LOG-TRANSFORMED AOV
# ===================================================

import numpy as np
import pandas as pd

# Copy model_df to avoid modifying the original dataset
model_reg = model_df.copy()

# -------------------------
# Check AOV validity
# -------------------------
print("===== AOV Validity Check =====")
print("Minimum AOV:", model_reg["AOV"].min())
print("Number of AOV <= 0:", (model_reg["AOV"] <= 0).sum())

# Since AOV is positive, we can use natural log
model_reg["log_AOV"] = np.log(model_reg["AOV"])


# -------------------------
# Define outlier variables
# -------------------------
outlier_vars = ["AOV", "BasketSize", "TotalQuantity", "AvgUnitPrice"]

# Use 99.5th percentile as upper threshold
upper_limits = model_reg[outlier_vars].quantile(0.995)

print("\n===== 99.5th Percentile Upper Limits =====")
display(upper_limits.to_frame("upper_limit").round(4))


# -------------------------
# Create outlier flag
# -------------------------
model_reg["is_outlier"] = False

for col in outlier_vars:
    model_reg[f"{col}_outlier"] = model_reg[col] > upper_limits[col]
    model_reg["is_outlier"] = model_reg["is_outlier"] | model_reg[f"{col}_outlier"]


# -------------------------
# Outlier summary
# -------------------------
print("\n===== Outlier Count by Variable =====")
outlier_summary = pd.DataFrame({
    "variable": outlier_vars,
    "upper_limit": [upper_limits[col] for col in outlier_vars],
    "outlier_count": [model_reg[f"{col}_outlier"].sum() for col in outlier_vars],
    "outlier_rate": [model_reg[f"{col}_outlier"].mean() for col in outlier_vars]
})

display(outlier_summary.round(4))

total_outliers = model_reg["is_outlier"].sum()
print("\nTotal unique outlier baskets:", total_outliers)
print("Total outlier rate:", round(model_reg["is_outlier"].mean(), 4))


# -------------------------
# Create cleaned regression dataset
# -------------------------
model_reg_clean = model_reg[model_reg["is_outlier"] == False].copy()

print("\n===== Shape Before and After Outlier Removal =====")
print("Original model_reg shape:", model_reg.shape)
print("Cleaned model_reg_clean shape:", model_reg_clean.shape)
print("Rows removed:", len(model_reg) - len(model_reg_clean))


# -------------------------
# Check rule_applied distribution before and after cleaning
# -------------------------
print("\n===== rule_applied Distribution BEFORE Cleaning =====")
display(model_reg["rule_applied"].value_counts().sort_index().to_frame("count"))

print("\n===== rule_applied Distribution AFTER Cleaning =====")
display(model_reg_clean["rule_applied"].value_counts().sort_index().to_frame("count"))

print("\nRule-applied rate before cleaning:", round(model_reg["rule_applied"].mean(), 4))
print("Rule-applied rate after cleaning:", round(model_reg_clean["rule_applied"].mean(), 4))


# -------------------------
# Compare descriptive statistics before and after cleaning
# -------------------------
print("\n===== Descriptive Statistics BEFORE Cleaning =====")
display(
    model_reg[
        ["AOV", "log_AOV", "BasketSize", "TotalQuantity", "AvgUnitPrice"]
    ].describe().round(4)
)

print("\n===== Descriptive Statistics AFTER Cleaning =====")
display(
    model_reg_clean[
        ["AOV", "log_AOV", "BasketSize", "TotalQuantity", "AvgUnitPrice"]
    ].describe().round(4)
)


# -------------------------
# Group comparison after cleaning
# -------------------------
print("\n===== Group Comparison by rule_applied AFTER Cleaning =====")
group_summary_clean = model_reg_clean.groupby("rule_applied").agg(
    baskets=("InvoiceNo", "count"),
    mean_AOV=("AOV", "mean"),
    median_AOV=("AOV", "median"),
    mean_log_AOV=("log_AOV", "mean"),
    median_log_AOV=("log_AOV", "median"),
    mean_BasketSize=("BasketSize", "mean"),
    median_BasketSize=("BasketSize", "median"),
    mean_TotalQuantity=("TotalQuantity", "mean"),
    median_TotalQuantity=("TotalQuantity", "median"),
    mean_AvgUnitPrice=("AvgUnitPrice", "mean"),
    median_AvgUnitPrice=("AvgUnitPrice", "median")
).round(4)

display(group_summary_clean)


# -------------------------
# Check top values after cleaning
# -------------------------
print("\n===== Top 10 Highest AOV Baskets AFTER Cleaning =====")
display(
    model_reg_clean.sort_values("AOV", ascending=False)[
        ["InvoiceNo", "AOV", "log_AOV", "rule_applied", "BasketSize", 
         "TotalQuantity", "AvgUnitPrice", "Country"]
    ].head(10)
)

print("\n===== Top 10 Largest BasketSize Baskets AFTER Cleaning =====")
display(
    model_reg_clean.sort_values("BasketSize", ascending=False)[
        ["InvoiceNo", "AOV", "log_AOV", "rule_applied", "BasketSize", 
         "TotalQuantity", "AvgUnitPrice", "Country"]
    ].head(10)
)

===== AOV Validity Check =====
Minimum AOV: 0.19
Number of AOV <= 0: 0

===== 99.5th Percentile Upper Limits =====


,upper_limit
AOV,5743.1000
BasketSize,284.0000
TotalQuantity,3566.5500
AvgUnitPrice,19.1657



===== Outlier Count by Variable =====


,variable,upper_limit,outlier_count,outlier_rate
0,AOV,5743.1000,198,0.0050
1,BasketSize,284.0000,197,0.0050
2,TotalQuantity,3566.5500,198,0.0050
3,AvgUnitPrice,19.1657,198,0.0050



Total unique outlier baskets: 647
Total outlier rate: 0.0164

===== Shape Before and After Outlier Removal =====
Original model_reg shape: (39516, 18)
Cleaned model_reg_clean shape: (38869, 18)
Rows removed: 647

===== rule_applied Distribution BEFORE Cleaning =====


...

,baskets,mean_AOV,median_AOV,mean_log_AOV,median_log_AOV,mean_BasketSize,median_BasketSize,mean_TotalQuantity,median_TotalQuantity,mean_AvgUnitPrice,median_AvgUnitPrice
rule_applied,,,,,,,,,,,
0,38488,412.6647,298.8500,5.4845,5.6999,22.6192,15.0000,230.9834,146.0000,3.2624,2.8297
1,381,707.9793,459.5000,6.2012,6.1301,56.9606,44.0000,389.4252,267.0000,2.7885,2.5324



===== Top 10 Highest AOV Baskets AFTER Cleaning =====


,InvoiceNo,AOV,log_AOV,rule_applied,BasketSize,TotalQuantity,AvgUnitPrice,Country
11642,517817,5737.3200,8.6547,0,17,2076,2.9265,United Kingdom
22701,543518,5735.2400,8.6544,0,48,2962,2.4087,Japan
32979,566931,5725.0000,8.6526,0,3,1250,4.5800,United Kingdom
12204,519196,5676.8100,8.6441,0,107,1360,5.2902,United Kingdom
33160,567385,5639.5200,8.6376,0,22,2472,3.1359,United Kingdom
1852,493994,5520.0000,8.6161,0,4,1200,4.6000,United Kingdom
27667,554845,5518.6800,8.6159,0,8,2140,2.1237,United Kingdom
24597,547812,5513.3200,8.6149,0,4,1336,4.3275,United Kingdom
29410,558776,5496.0000,8.6118,0,6,1200,4.5800,United Kingdom
37791,577747,5490.2400,8.6107,0,14,1182,6.6707,United Kingdom



===== Top 10 Largest BasketSize Baskets AFTER Cleaning =====


,InvoiceNo,AOV,log_AOV,rule_applied,BasketSize,TotalQuantity,AvgUnitPrice,Country
8498,510519,2415.8500,7.7898,0,284,496,4.8412,United Kingdom
33278,567671,1950.7700,7.5760,0,284,596,4.5194,United Kingdom
2035,494495,1992.1900,7.5970,0,283,536,4.5924,United Kingdom
34729,570871,1891.1200,7.5449,0,282,671,3.8481,United Kingdom
21883,541516,1582.8300,7.3670,0,281,609,3.1932,United Kingdom
26441,552232,2215.4100,7.7032,0,281,695,4.0902,United Kingdom
35435,572553,2026.7200,7.6142,0,279,651,4.0433,United Kingdom
33985,569246,2410.7400,7.7877,0,279,709,3.9248,United Kingdom
1619,493260,1976.4800,7.5891,0,278,397,5.2900,United Kingdom
21140,539492,2098.9600,7.6492,0,277,466,5.2073,United Kingdom


# Step 5: Baseline Regression — AOV ~ rule_applied

## Mục tiêu của bước này

Bước 5 nhằm chạy mô hình hồi quy baseline để kiểm tra mối quan hệ ban đầu giữa việc basket có chứa association rule và giá trị đơn hàng trung bình.

Rule được phân tích là:

```text
22748, 22745 → 22746
```

Biến chính được sử dụng là `rule_applied`:

* `rule_applied = 1` nếu basket chứa đầy đủ các sản phẩm trong rule.
* `rule_applied = 0` nếu basket không chứa đầy đủ rule.

Hai mô hình baseline được chạy:

```text
AOV ~ rule_applied
```

và:

```text
log_AOV ~ rule_applied
```

Mô hình thứ nhất sử dụng AOV gốc, còn mô hình thứ hai sử dụng log(AOV) để giảm ảnh hưởng của độ lệch trong phân phối doanh thu và cho phép diễn giải kết quả theo phần trăm.

## Dữ liệu sử dụng

Mô hình được chạy trên bộ dữ liệu đã xử lý outlier từ Step 4. Kích thước dữ liệu là:

```text
38,869 baskets
```

Phân phối biến `rule_applied` như sau:

| rule_applied | Số lượng basket |
| -----------: | --------------: |
|            0 |          38,488 |
|            1 |             381 |

Như vậy, sau khi xử lý outlier, vẫn còn 381 basket chứa đầy đủ rule. Số lượng này đủ để tiếp tục phân tích hồi quy.

## Kết quả mô hình AOV baseline

Mô hình đầu tiên có dạng:

```text
AOV = β0 + β1 * rule_applied + error
```

Kết quả chính:

| Chỉ số                       |  Giá trị |
| ---------------------------- | -------: |
| Intercept                    | 412.6647 |
| Coefficient của rule_applied | 295.3146 |
| p-value                      |   0.0000 |
| R-squared                    |   0.0033 |

Hệ số của `rule_applied` là 295.3146 và có ý nghĩa thống kê ở mức p-value nhỏ hơn 0.05.

Điều này có nghĩa là, trong mô hình baseline, các basket có chứa rule có AOV trung bình cao hơn khoảng 295.31 đơn vị so với các basket không chứa rule.

Cụ thể:

| Nhóm             | AOV ước lượng |
| ---------------- | ------------: |
| rule_applied = 0 |        412.66 |
| rule_applied = 1 |        707.98 |

Như vậy, ở mức so sánh thô, basket có rule có AOV cao hơn rõ rệt.

## Kết quả mô hình log(AOV) baseline

Mô hình thứ hai có dạng:

```text
log_AOV = β0 + β1 * rule_applied + error
```

Kết quả chính:

| Chỉ số                       | Giá trị |
| ---------------------------- | ------: |
| Coefficient của rule_applied |  0.7166 |
| p-value                      |  0.0000 |
| R-squared                    |  0.0035 |

Khi chuyển hệ số log sang phần trăm, kết quả cho thấy basket có rule_applied có AOV cao hơn khoảng:

```text
104.75%
```

so với basket không có rule, trước khi thêm các biến kiểm soát.

Khoảng tin cậy 95% của hiệu ứng phần trăm là từ khoảng 88.13% đến 122.83%.

## Nhận xét về ý nghĩa thống kê

Trong cả hai mô hình, hệ số của `rule_applied` đều có p-value rất nhỏ. Điều này cho thấy có sự khác biệt có ý nghĩa thống kê giữa nhóm basket có rule và nhóm basket không có rule.

Tuy nhiên, ý nghĩa thống kê không đồng nghĩa với tác động nhân quả. Kết quả này chỉ cho thấy basket có rule có xu hướng đi kèm với AOV cao hơn.

## Nhận xét về R-squared

R-squared của hai mô hình baseline chỉ khoảng 0.0033 đến 0.0035. Điều này có nghĩa là biến `rule_applied` một mình chỉ giải thích được khoảng 0.3% biến động của AOV.

Nói cách khác, mặc dù `rule_applied` có liên quan đến AOV cao hơn, AOV vẫn còn phụ thuộc vào nhiều yếu tố khác như:

* số lượng sản phẩm trong basket,
* tổng số lượng sản phẩm mua,
* đơn giá trung bình,
* quốc gia giao dịch,
* đặc điểm sản phẩm,
* hành vi mua hàng của khách hàng.

## Hạn chế của mô hình baseline

Mô hình baseline chưa kiểm soát các yếu tố gây nhiễu. Ở các bước trước, thống kê mô tả cho thấy nhóm `rule_applied = 1` có BasketSize lớn hơn nhiều so với nhóm `rule_applied = 0`.

Do đó, AOV cao hơn ở nhóm có rule có thể đến từ việc basket đó lớn hơn, chứ chưa chắc do bản thân rule tạo ra.

Vì vậy, chưa thể kết luận rằng rule gây ra tăng AOV. Kết quả ở bước này chỉ nên được hiểu là mối quan hệ ban đầu giữa rule và AOV.

## Kết luận bước 5

Baseline regression cho thấy các basket có chứa rule `22748, 22745 → 22746` có AOV cao hơn đáng kể so với các basket không chứa rule.

Tuy nhiên, do mô hình chưa kiểm soát BasketSize, TotalQuantity, AvgUnitPrice và Country, kết quả này chưa đủ để khẳng định tác động độc lập của rule lên AOV.

Bước tiếp theo cần thêm biến kiểm soát `BasketSize` để kiểm tra xem hiệu ứng của `rule_applied` còn tồn tại hay không sau khi kiểm soát quy mô giỏ hàng.


In [13]:
# =========================================
# STEP 5: BASELINE REGRESSION
# AOV ~ rule_applied
# log_AOV ~ rule_applied
# =========================================

import statsmodels.formula.api as smf
import numpy as np
import pandas as pd

# Use cleaned regression dataset from Step 4
reg_data = model_reg_clean.copy()

print("===== Regression Dataset Check =====")
print("Shape:", reg_data.shape)
print("rule_applied counts:")
display(reg_data["rule_applied"].value_counts().sort_index().to_frame("count"))


# -------------------------
# Model 1: Baseline linear AOV model
# -------------------------
model_1_aov = smf.ols(
    formula="AOV ~ rule_applied",
    data=reg_data
).fit(cov_type="HC3")   # robust standard errors

print("\n===== Model 1A: Baseline Linear Regression =====")
print("Formula: AOV ~ rule_applied")
print(model_1_aov.summary())


# -------------------------
# Model 1B: Baseline log AOV model
# -------------------------
model_1_log = smf.ols(
    formula="log_AOV ~ rule_applied",
    data=reg_data
).fit(cov_type="HC3")   # robust standard errors

print("\n===== Model 1B: Baseline Log-Linear Regression =====")
print("Formula: log_AOV ~ rule_applied")
print(model_1_log.summary())


# -------------------------
# Extract key results
# -------------------------
coef_aov = model_1_aov.params["rule_applied"]
pval_aov = model_1_aov.pvalues["rule_applied"]
ci_aov = model_1_aov.conf_int().loc["rule_applied"]

coef_log = model_1_log.params["rule_applied"]
pval_log = model_1_log.pvalues["rule_applied"]
ci_log = model_1_log.conf_int().loc["rule_applied"]

approx_pct_effect = (np.exp(coef_log) - 1) * 100
ci_log_pct = (np.exp(ci_log) - 1) * 100

baseline_summary = pd.DataFrame({
    "Model": [
        "Model 1A - AOV ~ rule_applied",
        "Model 1B - log_AOV ~ rule_applied"
    ],
    "Rule_Coefficient": [
        coef_aov,
        coef_log
    ],
    "P_Value": [
        pval_aov,
        pval_log
    ],
    "CI_Lower": [
        ci_aov[0],
        ci_log[0]
    ],
    "CI_Upper": [
        ci_aov[1],
        ci_log[1]
    ],
    "R_Squared": [
        model_1_aov.rsquared,
        model_1_log.rsquared
    ],
    "Adj_R_Squared": [
        model_1_aov.rsquared_adj,
        model_1_log.rsquared_adj
    ],
    "AIC": [
        model_1_aov.aic,
        model_1_log.aic
    ],
    "BIC": [
        model_1_aov.bic,
        model_1_log.bic
    ]
})

print("\n===== Baseline Regression Summary Table =====")
display(baseline_summary.round(6))

print("\n===== Interpretation Helpers =====")
print("Linear AOV coefficient:", round(coef_aov, 4))
print("This means rule_applied baskets have about", round(coef_aov, 4), "higher AOV on average before controls.")

print("\nLog AOV coefficient:", round(coef_log, 6))
print("Approximate percentage effect:", round(approx_pct_effect, 4), "%")

print("\n95% CI for log percentage effect:")
print("Lower:", round(ci_log_pct[0], 4), "%")
print("Upper:", round(ci_log_pct[1], 4), "%")

===== Regression Dataset Check =====
Shape: (38869, 18)
rule_applied counts:


,count
rule_applied,
0,38488
1,381



===== Model 1A: Baseline Linear Regression =====
Formula: AOV ~ rule_applied
                            OLS Regression Results                            
Dep. Variable:                    AOV   R-squared:                       0.003
Model:                            OLS   Adj. R-squared:                  0.003
Method:                 Least Squares   F-statistic:                     63.52
Date:                Thu, 04 Jun 2026   Prob (F-statistic):           1.63e-15
Time:                        14:37:22   Log-Likelihood:            -2.9729e+05
No. Observations:               38869   AIC:                         5.946e+05
Df Residuals:                   38867   BIC:                         5.946e+05
Df Model:                           1                                         
Covariance Type:                  HC3                                         
                   coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------

,Model,Rule_Coefficient,P_Value,CI_Lower,CI_Upper,R_Squared,Adj_R_Squared,AIC,BIC
0,Model 1A - AOV ~ rule_applied,295.3146,0.0000,222.6911,367.9381,0.0033,0.0032,594576.9682,594594.1041
1,Model 1B - log_AOV ~ rule_applied,0.7166,0.0000,0.6320,0.8013,0.0035,0.0035,123599.7111,123616.8470



===== Interpretation Helpers =====
Linear AOV coefficient: 295.3146
This means rule_applied baskets have about 295.3146 higher AOV on average before controls.

Log AOV coefficient: 0.716621
Approximate percentage effect: 104.7503 %

95% CI for log percentage effect:
Lower: 88.1342 %
Upper: 122.834 %


# Step 6: Hồi quy với biến kiểm soát BasketSize

## Mục tiêu của bước này

Bước 6 nhằm kiểm tra xem mối quan hệ giữa `rule_applied` và AOV có còn tồn tại sau khi kiểm soát quy mô giỏ hàng hay không.

Ở các bước thống kê mô tả trước đó, nhóm basket có `rule_applied = 1` có BasketSize lớn hơn rõ rệt so với nhóm `rule_applied = 0`. Vì vậy, AOV cao hơn ở nhóm có rule có thể đến từ việc các basket này vốn có nhiều sản phẩm hơn, chứ không nhất thiết do bản thân association rule tạo ra.

Do đó, bước này thêm biến kiểm soát `BasketSize` vào mô hình hồi quy.

Hai mô hình được chạy là:

```text
AOV ~ rule_applied + BasketSize
```

và:

```text
log_AOV ~ rule_applied + BasketSize
```

## Kết quả mô hình AOV với BasketSize control

Kết quả chính của mô hình `AOV ~ rule_applied + BasketSize` như sau:

| Biến         | Coefficient | p-value |
| ------------ | ----------: | ------: |
| rule_applied |     26.3143 |  0.4200 |
| BasketSize   |      7.8331 |  0.0000 |

Hệ số của `BasketSize` là 7.8331 và có ý nghĩa thống kê rất mạnh. Điều này cho thấy khi BasketSize tăng thêm 1 sản phẩm, AOV tăng trung bình khoảng 7.83 đơn vị, khi giữ các yếu tố khác trong mô hình không đổi.

Trong khi đó, hệ số của `rule_applied` giảm xuống còn 26.3143 và không còn có ý nghĩa thống kê do p-value bằng 0.4200.

Điều này cho thấy sau khi kiểm soát số lượng sản phẩm trong basket, chưa có đủ bằng chứng thống kê để kết luận rằng việc basket chứa rule làm AOV tăng trong mô hình AOV gốc.

## So sánh với mô hình baseline

Ở mô hình baseline trước đó, hệ số của `rule_applied` là:

```text
295.3146
```

Sau khi thêm `BasketSize`, hệ số này giảm xuống còn:

```text
26.3143
```

Mức giảm là:

```text
269.0002
```

Sự giảm mạnh này cho thấy phần lớn chênh lệch AOV ban đầu giữa nhóm có rule và nhóm không có rule đến từ sự khác biệt về BasketSize.

Nói cách khác, các basket có rule thường là các basket lớn hơn, nên AOV cao hơn là điều dễ xảy ra.

## Kết quả mô hình log(AOV)

Ở mô hình log-linear, hệ số của `rule_applied` cũng giảm mạnh.

| Model                    | Coefficient của rule_applied | Hiệu ứng phần trăm ước lượng |
| ------------------------ | ---------------------------: | ---------------------------: |
| Baseline log model       |                       0.7166 |                      104.75% |
| Log model với BasketSize |                       0.0863 |                        9.01% |

Sau khi kiểm soát BasketSize, hiệu ứng ước lượng giảm từ khoảng 104.75% xuống còn khoảng 9.01%.

p-value của `rule_applied` trong log model sau khi thêm BasketSize là 0.0429. Điều này cho thấy hiệu ứng vẫn có ý nghĩa thống kê ở mức 5%, nhưng mức ý nghĩa khá sát ngưỡng. Vì vậy, kết quả này nên được diễn giải thận trọng.

## R-squared

R-squared của mô hình AOV tăng từ 0.0033 ở baseline lên 0.1972 sau khi thêm BasketSize.

Điều này cho thấy BasketSize giải thích biến động AOV tốt hơn rất nhiều so với chỉ dùng biến `rule_applied`.

Tương tự, R-squared của log model cũng tăng từ 0.0035 lên 0.1983.

## Nhận xét

Kết quả Step 6 cho thấy BasketSize là một biến kiểm soát rất quan trọng. Mối quan hệ mạnh giữa `rule_applied` và AOV ở mô hình baseline bị giảm mạnh sau khi thêm BasketSize.

Trong mô hình AOV gốc, `rule_applied` không còn có ý nghĩa thống kê. Trong mô hình log(AOV), `rule_applied` vẫn còn liên quan đến AOV cao hơn khoảng 9.01%, nhưng hiệu ứng nhỏ hơn nhiều và cần được diễn giải thận trọng.

## Kết luận bước 6

Kết quả cho thấy phần lớn hiệu ứng ban đầu của association rule lên AOV có thể được giải thích bởi BasketSize. Nói cách khác, các basket chứa rule thường là các basket lớn hơn, nên AOV cao hơn không hoàn toàn đến từ bản thân rule.

Bước tiếp theo cần thêm biến kiểm soát về giá, cụ thể là `AvgUnitPrice`, để kiểm tra xem hiệu ứng của `rule_applied` còn tồn tại sau khi kiểm soát cả quy mô giỏ hàng và mức giá trung bình hay không.


In [14]:
# =========================================
# STEP 6: REGRESSION WITH BASKETSIZE CONTROL
# AOV ~ rule_applied + BasketSize
# log_AOV ~ rule_applied + BasketSize
# =========================================

import statsmodels.formula.api as smf
import numpy as np
import pandas as pd

# Use cleaned regression dataset
reg_data = model_reg_clean.copy()

print("===== Regression Dataset Check =====")
print("Shape:", reg_data.shape)
print("rule_applied counts:")
display(reg_data["rule_applied"].value_counts().sort_index().to_frame("count"))


# -------------------------
# Refit baseline models for comparison
# -------------------------
baseline_aov = smf.ols(
    formula="AOV ~ rule_applied",
    data=reg_data
).fit(cov_type="HC3")

baseline_log = smf.ols(
    formula="log_AOV ~ rule_applied",
    data=reg_data
).fit(cov_type="HC3")


# -------------------------
# Model 2A: AOV with BasketSize control
# -------------------------
model_2_aov = smf.ols(
    formula="AOV ~ rule_applied + BasketSize",
    data=reg_data
).fit(cov_type="HC3")

print("\n===== Model 2A: Linear Regression with BasketSize Control =====")
print("Formula: AOV ~ rule_applied + BasketSize")
print(model_2_aov.summary())


# -------------------------
# Model 2B: log_AOV with BasketSize control
# -------------------------
model_2_log = smf.ols(
    formula="log_AOV ~ rule_applied + BasketSize",
    data=reg_data
).fit(cov_type="HC3")

print("\n===== Model 2B: Log-Linear Regression with BasketSize Control =====")
print("Formula: log_AOV ~ rule_applied + BasketSize")
print(model_2_log.summary())


# -------------------------
# Build comparison table
# -------------------------
def extract_model_info(model, model_name):
    coef = model.params["rule_applied"]
    pval = model.pvalues["rule_applied"]
    ci = model.conf_int().loc["rule_applied"]

    return {
        "Model": model_name,
        "Rule_Coefficient": coef,
        "P_Value": pval,
        "CI_Lower": ci[0],
        "CI_Upper": ci[1],
        "R_Squared": model.rsquared,
        "Adj_R_Squared": model.rsquared_adj,
        "AIC": model.aic,
        "BIC": model.bic
    }

model_comparison_step6 = pd.DataFrame([
    extract_model_info(baseline_aov, "Model 1A - AOV ~ rule_applied"),
    extract_model_info(model_2_aov, "Model 2A - AOV ~ rule_applied + BasketSize"),
    extract_model_info(baseline_log, "Model 1B - log_AOV ~ rule_applied"),
    extract_model_info(model_2_log, "Model 2B - log_AOV ~ rule_applied + BasketSize")
])

print("\n===== Model Comparison Table =====")
display(model_comparison_step6.round(6))


# -------------------------
# Percentage interpretation for log models
# -------------------------
baseline_log_coef = baseline_log.params["rule_applied"]
model_2_log_coef = model_2_log.params["rule_applied"]

baseline_pct_effect = (np.exp(baseline_log_coef) - 1) * 100
model_2_pct_effect = (np.exp(model_2_log_coef) - 1) * 100

print("\n===== Percentage Effect Comparison from Log Models =====")
print("Baseline log coefficient:", round(baseline_log_coef, 6))
print("Baseline approximate percentage effect:", round(baseline_pct_effect, 4), "%")

print("\nBasketSize-control log coefficient:", round(model_2_log_coef, 6))
print("BasketSize-control approximate percentage effect:", round(model_2_pct_effect, 4), "%")


# -------------------------
# Coefficient change
# -------------------------
aov_coef_change = model_2_aov.params["rule_applied"] - baseline_aov.params["rule_applied"]
log_coef_change = model_2_log.params["rule_applied"] - baseline_log.params["rule_applied"]

print("\n===== Change in rule_applied Coefficient After Adding BasketSize =====")
print("AOV coefficient before control:", round(baseline_aov.params["rule_applied"], 4))
print("AOV coefficient after BasketSize control:", round(model_2_aov.params["rule_applied"], 4))
print("AOV coefficient change:", round(aov_coef_change, 4))

print("\nlog_AOV coefficient before control:", round(baseline_log.params["rule_applied"], 6))
print("log_AOV coefficient after BasketSize control:", round(model_2_log.params["rule_applied"], 6))
print("log_AOV coefficient change:", round(log_coef_change, 6))

===== Regression Dataset Check =====
Shape: (38869, 18)
rule_applied counts:


,count
rule_applied,
0,38488
1,381



===== Model 2A: Linear Regression with BasketSize Control =====
Formula: AOV ~ rule_applied + BasketSize
                            OLS Regression Results                            
Dep. Variable:                    AOV   R-squared:                       0.197
Model:                            OLS   Adj. R-squared:                  0.197
Method:                 Least Squares   F-statistic:                     2111.
Date:                Thu, 04 Jun 2026   Prob (F-statistic):               0.00
Time:                        14:39:13   Log-Likelihood:            -2.9308e+05
No. Observations:               38869   AIC:                         5.862e+05
Df Residuals:                   38866   BIC:                         5.862e+05
Df Model:                           2                                         
Covariance Type:                  HC3                                         
                   coef    std err          z      P>|z|      [0.025      0.975]
-----------------------

,Model,Rule_Coefficient,P_Value,CI_Lower,CI_Upper,R_Squared,Adj_R_Squared,AIC,BIC
0,Model 1A - AOV ~ rule_applied,295.3146,0.0000,222.6911,367.9381,0.0033,0.0032,594576.9682,594594.1041
1,Model 2A - AOV ~ rule_applied + BasketSize,26.3144,0.4200,-37.6436,90.2723,0.1972,0.1972,586167.1961,586192.9000
2,Model 1B - log_AOV ~ rule_applied,0.7166,0.0000,0.6320,0.8013,0.0035,0.0035,123599.7111,123616.8470
3,Model 2B - log_AOV ~ rule_applied + BasketSize,0.0863,0.0429,0.0028,0.1698,0.1983,0.1983,115145.8524,115171.5563



===== Percentage Effect Comparison from Log Models =====
Baseline log coefficient: 0.716621
Baseline approximate percentage effect: 104.7503 %

BasketSize-control log coefficient: 0.086263
BasketSize-control approximate percentage effect: 9.0093 %

===== Change in rule_applied Coefficient After Adding BasketSize =====
AOV coefficient before control: 295.3146
AOV coefficient after BasketSize control: 26.3143
AOV coefficient change: -269.0002

log_AOV coefficient before control: 0.716621
log_AOV coefficient after BasketSize control: 0.086263
log_AOV coefficient change: -0.630358


# Step 7: Hồi quy với biến kiểm soát BasketSize và AvgUnitPrice

## Mục tiêu của bước này

Bước 7 nhằm kiểm tra xem mối quan hệ giữa `rule_applied` và AOV có còn tồn tại sau khi kiểm soát cả quy mô giỏ hàng và mức giá trung bình của sản phẩm hay không.

Ở yêu cầu ban đầu, mô hình regression cần có các biến kiểm soát như price và promotion. Trong bộ dữ liệu hiện tại không có biến promotion, do đó chưa thể kiểm soát trực tiếp yếu tố khuyến mãi. Vì vậy, biến `AvgUnitPrice` được sử dụng làm biến kiểm soát liên quan đến giá.

Hai mô hình được chạy trong bước này là:

```text
AOV ~ rule_applied + BasketSize + AvgUnitPrice
```

và:

```text
log_AOV ~ rule_applied + BasketSize + AvgUnitPrice
```

## Kết quả mô hình AOV

Kết quả chính của mô hình `AOV ~ rule_applied + BasketSize + AvgUnitPrice` như sau:

| Biến         | Coefficient | p-value |
| ------------ | ----------: | ------: |
| rule_applied |     28.1493 |  0.3878 |
| BasketSize   |      7.8362 |  0.0000 |
| AvgUnitPrice |      4.0969 |  0.0000 |

Hệ số của `rule_applied` là 28.1493. Điều này có nghĩa là sau khi kiểm soát BasketSize và AvgUnitPrice, các basket có chứa rule có AOV cao hơn khoảng 28.15 đơn vị so với các basket không chứa rule.

Tuy nhiên, p-value của `rule_applied` là 0.3878, lớn hơn mức ý nghĩa 0.05. Vì vậy, trong mô hình AOV gốc, chưa có đủ bằng chứng thống kê để kết luận rằng `rule_applied` có liên hệ độc lập với AOV.

Ngược lại, `BasketSize` và `AvgUnitPrice` đều có ý nghĩa thống kê. Điều này cho thấy AOV chịu ảnh hưởng rõ rệt bởi số lượng sản phẩm khác nhau trong basket và mức giá trung bình của sản phẩm.

## So sánh với các mô hình trước

Kết quả so sánh hệ số của `rule_applied` qua các mô hình AOV như sau:

| Model                                                    | Coefficient của rule_applied | p-value | R-squared |
| -------------------------------------------------------- | ---------------------------: | ------: | --------: |
| Model 1A: AOV ~ rule_applied                             |                     295.3146 |  0.0000 |    0.0033 |
| Model 2A: AOV ~ rule_applied + BasketSize                |                      26.3143 |  0.4200 |    0.1972 |
| Model 3A: AOV ~ rule_applied + BasketSize + AvgUnitPrice |                      28.1493 |  0.3878 |    0.1975 |

Hệ số của `rule_applied` giảm mạnh từ 295.3146 ở mô hình baseline xuống còn khoảng 26–28 sau khi thêm BasketSize. Khi thêm AvgUnitPrice, hệ số này gần như không thay đổi.

Điều này cho thấy sự sụt giảm lớn của hiệu ứng rule chủ yếu đến từ việc kiểm soát BasketSize, chứ không phải từ AvgUnitPrice.

## Kết quả mô hình log(AOV)

Ở mô hình log-linear, hiệu ứng của `rule_applied` qua các mô hình như sau:

| Model                                                        | Coefficient của rule_applied | Hiệu ứng phần trăm ước lượng |
| ------------------------------------------------------------ | ---------------------------: | ---------------------------: |
| Model 1B: log_AOV ~ rule_applied                             |                       0.7166 |                      104.75% |
| Model 2B: log_AOV ~ rule_applied + BasketSize                |                       0.0863 |                        9.01% |
| Model 3B: log_AOV ~ rule_applied + BasketSize + AvgUnitPrice |                       0.0858 |                        8.96% |

Sau khi kiểm soát BasketSize và AvgUnitPrice, basket có `rule_applied = 1` được ước lượng là có AOV cao hơn khoảng 8.96% so với basket không có rule.

Tuy nhiên, p-value của `rule_applied` trong mô hình log(AOV) là 0.0442, chỉ vừa thấp hơn mức 0.05. Vì vậy, kết quả này nên được diễn giải thận trọng. Có thể nói rằng mô hình log-linear cho thấy một mối liên hệ dương nhỏ giữa rule và AOV, nhưng bằng chứng thống kê không quá mạnh.

## Vai trò của AvgUnitPrice

Biến `AvgUnitPrice` có ý nghĩa thống kê trong mô hình AOV, với hệ số dương. Điều này cho thấy các basket có mức giá trung bình cao hơn thường có AOV cao hơn.

Tuy nhiên, việc thêm `AvgUnitPrice` không làm thay đổi đáng kể hệ số của `rule_applied`. Điều này cho thấy mức giá trung bình không phải là yếu tố chính giải thích mối quan hệ ban đầu giữa rule và AOV. Yếu tố quan trọng hơn vẫn là BasketSize.

## Nhận xét

Kết quả Step 7 củng cố nhận định từ Step 6 rằng BasketSize là biến gây nhiễu quan trọng nhất. Mối quan hệ mạnh giữa `rule_applied` và AOV trong mô hình baseline phần lớn biến mất sau khi kiểm soát quy mô giỏ hàng.

Trong mô hình AOV gốc, `rule_applied` không còn có ý nghĩa thống kê sau khi thêm các biến kiểm soát. Trong mô hình log(AOV), `rule_applied` vẫn có mối liên hệ dương nhỏ khoảng 8.96%, nhưng kết quả này khá sát ngưỡng ý nghĩa thống kê và cần được diễn giải cẩn thận.

## Kết luận bước 7

Sau khi kiểm soát BasketSize và AvgUnitPrice, chưa có bằng chứng mạnh cho thấy association rule có tác động độc lập đáng kể lên AOV trong mô hình AOV gốc.

Kết quả hiện tại cho thấy các basket có rule thường có AOV cao hơn chủ yếu vì chúng là các basket lớn hơn. Vì vậy, bước tiếp theo cần kiểm tra thêm biến `TotalQuantity` để xem liệu tổng số lượng sản phẩm mua có tiếp tục giải thích phần còn lại của sự khác biệt AOV hay không.


In [15]:
# ==========================================================
# STEP 7: REGRESSION WITH BASKETSIZE AND AVGUNITPRICE CONTROL
# AOV ~ rule_applied + BasketSize + AvgUnitPrice
# log_AOV ~ rule_applied + BasketSize + AvgUnitPrice
# ==========================================================

import statsmodels.formula.api as smf
import numpy as np
import pandas as pd

# Use cleaned regression dataset
reg_data = model_reg_clean.copy()

print("===== Regression Dataset Check =====")
print("Shape:", reg_data.shape)
print("rule_applied counts:")
display(reg_data["rule_applied"].value_counts().sort_index().to_frame("count"))

print("\n===== Control Variables Check =====")
display(
    reg_data[["AOV", "log_AOV", "rule_applied", "BasketSize", "AvgUnitPrice"]]
    .describe()
    .round(4)
)


# -------------------------
# Refit previous models for comparison
# -------------------------
model_1_aov = smf.ols(
    formula="AOV ~ rule_applied",
    data=reg_data
).fit(cov_type="HC3")

model_2_aov = smf.ols(
    formula="AOV ~ rule_applied + BasketSize",
    data=reg_data
).fit(cov_type="HC3")

model_1_log = smf.ols(
    formula="log_AOV ~ rule_applied",
    data=reg_data
).fit(cov_type="HC3")

model_2_log = smf.ols(
    formula="log_AOV ~ rule_applied + BasketSize",
    data=reg_data
).fit(cov_type="HC3")


# -------------------------
# Model 3A: AOV with BasketSize and AvgUnitPrice controls
# -------------------------
model_3_aov = smf.ols(
    formula="AOV ~ rule_applied + BasketSize + AvgUnitPrice",
    data=reg_data
).fit(cov_type="HC3")

print("\n===== Model 3A: Linear Regression with BasketSize and AvgUnitPrice Controls =====")
print("Formula: AOV ~ rule_applied + BasketSize + AvgUnitPrice")
print(model_3_aov.summary())


# -------------------------
# Model 3B: log_AOV with BasketSize and AvgUnitPrice controls
# -------------------------
model_3_log = smf.ols(
    formula="log_AOV ~ rule_applied + BasketSize + AvgUnitPrice",
    data=reg_data
).fit(cov_type="HC3")

print("\n===== Model 3B: Log-Linear Regression with BasketSize and AvgUnitPrice Controls =====")
print("Formula: log_AOV ~ rule_applied + BasketSize + AvgUnitPrice")
print(model_3_log.summary())


# -------------------------
# Build comparison table
# -------------------------
def extract_model_info(model, model_name):
    coef = model.params["rule_applied"]
    pval = model.pvalues["rule_applied"]
    ci = model.conf_int().loc["rule_applied"]

    return {
        "Model": model_name,
        "Rule_Coefficient": coef,
        "P_Value": pval,
        "CI_Lower": ci[0],
        "CI_Upper": ci[1],
        "R_Squared": model.rsquared,
        "Adj_R_Squared": model.rsquared_adj,
        "AIC": model.aic,
        "BIC": model.bic
    }

model_comparison_step7 = pd.DataFrame([
    extract_model_info(model_1_aov, "Model 1A - AOV ~ rule_applied"),
    extract_model_info(model_2_aov, "Model 2A - AOV ~ rule_applied + BasketSize"),
    extract_model_info(model_3_aov, "Model 3A - AOV ~ rule_applied + BasketSize + AvgUnitPrice"),
    extract_model_info(model_1_log, "Model 1B - log_AOV ~ rule_applied"),
    extract_model_info(model_2_log, "Model 2B - log_AOV ~ rule_applied + BasketSize"),
    extract_model_info(model_3_log, "Model 3B - log_AOV ~ rule_applied + BasketSize + AvgUnitPrice")
])

print("\n===== Model Comparison Table =====")
display(model_comparison_step7.round(6))


# -------------------------
# Percentage interpretation for log models
# -------------------------
model_1_log_coef = model_1_log.params["rule_applied"]
model_2_log_coef = model_2_log.params["rule_applied"]
model_3_log_coef = model_3_log.params["rule_applied"]

model_1_pct_effect = (np.exp(model_1_log_coef) - 1) * 100
model_2_pct_effect = (np.exp(model_2_log_coef) - 1) * 100
model_3_pct_effect = (np.exp(model_3_log_coef) - 1) * 100

print("\n===== Percentage Effect Comparison from Log Models =====")
print("Model 1B baseline percentage effect:", round(model_1_pct_effect, 4), "%")
print("Model 2B with BasketSize percentage effect:", round(model_2_pct_effect, 4), "%")
print("Model 3B with BasketSize + AvgUnitPrice percentage effect:", round(model_3_pct_effect, 4), "%")


# -------------------------
# Coefficient change
# -------------------------
print("\n===== Change in rule_applied Coefficient Across Models =====")

print("AOV coefficient:")
print("Model 1A:", round(model_1_aov.params["rule_applied"], 4))
print("Model 2A:", round(model_2_aov.params["rule_applied"], 4))
print("Model 3A:", round(model_3_aov.params["rule_applied"], 4))

print("\nlog_AOV coefficient:")
print("Model 1B:", round(model_1_log.params["rule_applied"], 6))
print("Model 2B:", round(model_2_log.params["rule_applied"], 6))
print("Model 3B:", round(model_3_log.params["rule_applied"], 6))


# -------------------------
# Save models for later steps
# -------------------------
models_step7 = {
    "model_1_aov": model_1_aov,
    "model_2_aov": model_2_aov,
    "model_3_aov": model_3_aov,
    "model_1_log": model_1_log,
    "model_2_log": model_2_log,
    "model_3_log": model_3_log
}

===== Regression Dataset Check =====
Shape: (38869, 18)
rule_applied counts:


,count
rule_applied,
0,38488
1,381



===== Control Variables Check =====


,AOV,log_AOV,rule_applied,BasketSize,AvgUnitPrice
count,38869.0000,38869.0000,38869.0000,38869.0000,38869.0000
mean,415.5594,5.4916,0.0098,22.9558,3.2578
std,508.3327,1.1886,0.0985,28.7801,1.9329
min,0.1900,-1.6607,0.0000,1.0000,0.0400
25%,149.2300,5.0055,0.0000,6.0000,2.0302
50%,299.9200,5.7035,0.0000,15.0000,2.8250
75%,473.2000,6.1595,0.0000,28.0000,3.9618
max,5737.3200,8.6547,1.0000,284.0000,19.1625



===== Model 3A: Linear Regression with BasketSize and AvgUnitPrice Controls =====
Formula: AOV ~ rule_applied + BasketSize + AvgUnitPrice
                            OLS Regression Results                            
Dep. Variable:                    AOV   R-squared:                       0.197
Model:                            OLS   Adj. R-squared:                  0.197
Method:                 Least Squares   F-statistic:                     1434.
Date:                Thu, 04 Jun 2026   Prob (F-statistic):               0.00
Time:                        14:41:13   Log-Likelihood:            -2.9307e+05
No. Observations:               38869   AIC:                         5.862e+05
Df Residuals:                   38865   BIC:                         5.862e+05
Df Model:                           3                                         
Covariance Type:                  HC3                                         
                   coef    std err          z      P>|z|      [0.025   

,Model,Rule_Coefficient,P_Value,CI_Lower,CI_Upper,R_Squared,Adj_R_Squared,AIC,BIC
0,Model 1A - AOV ~ rule_applied,295.3146,0.0000,222.6911,367.9381,0.0033,0.0032,594576.9682,594594.1041
1,Model 2A - AOV ~ rule_applied + BasketSize,26.3144,0.4200,-37.6436,90.2723,0.1972,0.1972,586167.1961,586192.9000
2,Model 3A - AOV ~ rule_applied + BasketSize + A...,28.1493,0.3878,-35.7302,92.0288,0.1975,0.1974,586157.4530,586191.7248
3,Model 1B - log_AOV ~ rule_applied,0.7166,0.0000,0.6320,0.8013,0.0035,0.0035,123599.7111,123616.8470
4,Model 2B - log_AOV ~ rule_applied + BasketSize,0.0863,0.0429,0.0028,0.1698,0.1983,0.1983,115145.8524,115171.5563
5,Model 3B - log_AOV ~ rule_applied + BasketSize...,0.0858,0.0442,0.0022,0.1694,0.1984,0.1983,115147.7211,115181.9929



===== Percentage Effect Comparison from Log Models =====
Model 1B baseline percentage effect: 104.7503 %
Model 2B with BasketSize percentage effect: 9.0093 %
Model 3B with BasketSize + AvgUnitPrice percentage effect: 8.9599 %

===== Change in rule_applied Coefficient Across Models =====
AOV coefficient:
Model 1A: 295.3146
Model 2A: 26.3143
Model 3A: 28.1493

log_AOV coefficient:
Model 1B: 0.716621
Model 2B: 0.086263
Model 3B: 0.08581


# Step 8: Hồi quy với BasketSize, AvgUnitPrice và TotalQuantity

## Mục tiêu của bước này

Bước 8 nhằm bổ sung biến `TotalQuantity` vào mô hình hồi quy để kiểm soát thêm quy mô mua hàng của từng basket.

Ở các bước trước, mô hình đã kiểm soát:

* `BasketSize`: số lượng sản phẩm khác nhau trong basket.
* `AvgUnitPrice`: đơn giá trung bình của sản phẩm trong basket.

Tuy nhiên, BasketSize chỉ phản ánh số lượng loại sản phẩm khác nhau, chưa phản ánh tổng số lượng đơn vị sản phẩm được mua. Vì vậy, bước này thêm biến `TotalQuantity` để kiểm soát tổng số lượng sản phẩm trong basket.

Hai mô hình được chạy là:

```text
AOV ~ rule_applied + BasketSize + AvgUnitPrice + TotalQuantity
```

và:

```text
log_AOV ~ rule_applied + BasketSize + AvgUnitPrice + TotalQuantity
```

## Kiểm tra tương quan giữa các biến

Kết quả correlation matrix cho thấy `TotalQuantity` có tương quan rất mạnh với AOV:

| Cặp biến                    | Correlation |
| --------------------------- | ----------: |
| AOV và TotalQuantity        |      0.7974 |
| log_AOV và TotalQuantity    |      0.5957 |
| AOV và BasketSize           |      0.4441 |
| BasketSize và TotalQuantity |      0.2899 |
| rule_applied và AOV         |      0.0572 |

Tương quan giữa `AOV` và `TotalQuantity` là 0.7974, cho thấy tổng số lượng sản phẩm mua là một yếu tố giải thích rất mạnh cho giá trị đơn hàng.

Trong khi đó, tương quan giữa `rule_applied` và `AOV` chỉ là 0.0572, khá yếu. Điều này cho thấy mối liên hệ trực tiếp giữa rule và AOV không mạnh nếu chỉ xét tương quan tuyến tính đơn giản.

Tương quan giữa `BasketSize` và `TotalQuantity` là 0.2899, không quá cao. Điều này cho thấy hai biến này không hoàn toàn trùng lặp: `BasketSize` đo số loại sản phẩm khác nhau, còn `TotalQuantity` đo tổng số đơn vị sản phẩm được mua.

## Kết quả mô hình AOV

Kết quả chính của mô hình `AOV ~ rule_applied + BasketSize + AvgUnitPrice + TotalQuantity` như sau:

| Biến          | Coefficient | p-value |
| ------------- | ----------: | ------: |
| rule_applied  |    -19.5486 |  0.3428 |
| BasketSize    |      3.9051 |  0.0000 |
| AvgUnitPrice  |     51.8606 |  0.0000 |
| TotalQuantity |      1.2959 |  0.0000 |

Sau khi thêm `TotalQuantity`, hệ số của `rule_applied` chuyển thành -19.5486 và không có ý nghĩa thống kê vì p-value bằng 0.3428.

Điều này cho thấy sau khi kiểm soát số loại sản phẩm, đơn giá trung bình và tổng số lượng sản phẩm mua, chưa có bằng chứng thống kê cho thấy basket có chứa rule sẽ có AOV cao hơn.

Ngược lại, các biến `BasketSize`, `AvgUnitPrice` và `TotalQuantity` đều có ý nghĩa thống kê rất mạnh. Điều này cho thấy AOV bị ảnh hưởng rõ rệt bởi quy mô và giá trị của basket.

## So sánh hệ số rule_applied qua các mô hình AOV

| Model                                                 | Coefficient của rule_applied | p-value | R-squared |
| ----------------------------------------------------- | ---------------------------: | ------: | --------: |
| Model 1A: AOV ~ rule_applied                          |                     295.3146 |  0.0000 |    0.0033 |
| Model 2A: + BasketSize                                |                      26.3143 |  0.4200 |    0.1972 |
| Model 3A: + BasketSize + AvgUnitPrice                 |                      28.1493 |  0.3878 |    0.1975 |
| Model 4A: + BasketSize + AvgUnitPrice + TotalQuantity |                     -19.5486 |  0.3428 |    0.7220 |

Ở mô hình baseline, hệ số của `rule_applied` là 295.3146 và có ý nghĩa thống kê. Tuy nhiên, khi thêm các biến kiểm soát, đặc biệt là `BasketSize` và `TotalQuantity`, hệ số này giảm mạnh và không còn có ý nghĩa thống kê.

R-squared tăng từ 0.0033 ở mô hình baseline lên 0.7220 ở mô hình đầy đủ. Điều này cho thấy các biến kiểm soát, đặc biệt là `TotalQuantity`, giải thích phần lớn biến động của AOV.

## Kết quả mô hình log(AOV)

Ở mô hình log-linear, hiệu ứng của `rule_applied` cũng giảm mạnh qua từng bước:

| Model                                                 | Coefficient của rule_applied | Hiệu ứng phần trăm ước lượng | p-value |
| ----------------------------------------------------- | ---------------------------: | ---------------------------: | ------: |
| Model 1B: log_AOV ~ rule_applied                      |                       0.7166 |                      104.75% |  0.0000 |
| Model 2B: + BasketSize                                |                       0.0863 |                        9.01% |  0.0429 |
| Model 3B: + BasketSize + AvgUnitPrice                 |                       0.0858 |                        8.96% |  0.0442 |
| Model 4B: + BasketSize + AvgUnitPrice + TotalQuantity |                       0.0085 |                        0.85% |  0.8154 |

Sau khi thêm `TotalQuantity`, hiệu ứng ước lượng của `rule_applied` chỉ còn khoảng 0.85% và hoàn toàn không có ý nghĩa thống kê.

Điều này củng cố kết luận rằng mối quan hệ dương giữa rule và AOV ở các mô hình đơn giản chủ yếu đến từ sự khác biệt về quy mô basket, không phải từ bản thân rule.

## Nhận xét về vai trò của TotalQuantity

`TotalQuantity` là biến kiểm soát rất quan trọng vì nó có liên quan trực tiếp đến tổng doanh thu của basket. Khi khách hàng mua nhiều đơn vị sản phẩm hơn, AOV thường tăng lên.

Sau khi thêm `TotalQuantity`, mô hình giải thích được 72.2% biến động của AOV. Đây là mức cải thiện rất lớn so với các mô hình trước đó.

Tuy nhiên, do `TotalQuantity` có mối liên hệ rất mạnh với AOV, cần diễn giải kết quả một cách cẩn thận. Trong bối cảnh này, `TotalQuantity` đóng vai trò là biến kiểm soát quy mô mua hàng, giúp tách ảnh hưởng của association rule khỏi ảnh hưởng của số lượng sản phẩm được mua.

## Nhận xét về cảnh báo multicollinearity

Kết quả model có cảnh báo về condition number lớn. Điều này có thể đến từ việc các biến có thang đo khác nhau, ví dụ `TotalQuantity` có giá trị lớn hơn nhiều so với `AvgUnitPrice`, hoặc có thể phản ánh một phần vấn đề đa cộng tuyến.

Tuy nhiên, tương quan giữa `BasketSize` và `TotalQuantity` chỉ là 0.2899, không quá cao. Vì vậy, chưa thể kết luận có đa cộng tuyến nghiêm trọng chỉ dựa vào cảnh báo này. Cần kiểm tra thêm VIF ở bước tiếp theo để đánh giá rõ hơn.

## Kết luận bước 8

Sau khi kiểm soát BasketSize, AvgUnitPrice và TotalQuantity, biến `rule_applied` không còn có ý nghĩa thống kê trong cả mô hình AOV và mô hình log(AOV).

Kết quả cho thấy AOV cao hơn ở nhóm basket có rule chủ yếu được giải thích bởi việc các basket này có quy mô lớn hơn và tổng số lượng mua cao hơn. Chưa có bằng chứng mạnh cho thấy bản thân association rule tạo ra tác động độc lập đáng kể lên AOV.

Bước tiếp theo nên kiểm tra đa cộng tuyến bằng VIF trước khi tiếp tục thêm các biến kiểm soát khác như Country.


In [18]:
# ===============================================================
# STEP 8: REGRESSION WITH BASKETSIZE, AVGUNITPRICE, TOTALQUANTITY
# AOV ~ rule_applied + BasketSize + AvgUnitPrice + TotalQuantity
# log_AOV ~ rule_applied + BasketSize + AvgUnitPrice + TotalQuantity
# ===============================================================

import statsmodels.formula.api as smf
import numpy as np
import pandas as pd

# Use cleaned regression dataset
reg_data = model_reg_clean.copy()

print("===== Regression Dataset Check =====")
print("Shape:", reg_data.shape)
print("rule_applied counts:")
display(reg_data["rule_applied"].value_counts().sort_index().to_frame("count"))


# -------------------------
# Correlation check
# -------------------------
print("\n===== Correlation Matrix: Main Variables =====")
corr_vars = ["AOV", "log_AOV", "rule_applied", "BasketSize", "AvgUnitPrice", "TotalQuantity"]

corr_matrix = reg_data[corr_vars].corr().round(4)
display(corr_matrix)


# -------------------------
# Descriptive check by rule_applied
# -------------------------
print("\n===== Group Comparison Before Step 8 Regression =====")
group_check_step8 = reg_data.groupby("rule_applied").agg(
    baskets=("InvoiceNo", "count"),
    mean_AOV=("AOV", "mean"),
    median_AOV=("AOV", "median"),
    mean_BasketSize=("BasketSize", "mean"),
    median_BasketSize=("BasketSize", "median"),
    mean_AvgUnitPrice=("AvgUnitPrice", "mean"),
    median_AvgUnitPrice=("AvgUnitPrice", "median"),
    mean_TotalQuantity=("TotalQuantity", "mean"),
    median_TotalQuantity=("TotalQuantity", "median")
).round(4)

display(group_check_step8)


# -------------------------
# Refit previous models for comparison
# -------------------------
model_1_aov = smf.ols(
    formula="AOV ~ rule_applied",
    data=reg_data
).fit(cov_type="HC3")

model_2_aov = smf.ols(
    formula="AOV ~ rule_applied + BasketSize",
    data=reg_data
).fit(cov_type="HC3")

model_3_aov = smf.ols(
    formula="AOV ~ rule_applied + BasketSize + AvgUnitPrice",
    data=reg_data
).fit(cov_type="HC3")

model_1_log = smf.ols(
    formula="log_AOV ~ rule_applied",
    data=reg_data
).fit(cov_type="HC3")

model_2_log = smf.ols(
    formula="log_AOV ~ rule_applied + BasketSize",
    data=reg_data
).fit(cov_type="HC3")

model_3_log = smf.ols(
    formula="log_AOV ~ rule_applied + BasketSize + AvgUnitPrice",
    data=reg_data
).fit(cov_type="HC3")


# -------------------------
# Model 4A: AOV with BasketSize, AvgUnitPrice, TotalQuantity controls
# -------------------------
model_4_aov = smf.ols(
    formula="AOV ~ rule_applied + BasketSize + AvgUnitPrice + TotalQuantity",
    data=reg_data
).fit(cov_type="HC3")

print("\n===== Model 4A: Linear Regression with BasketSize, AvgUnitPrice, and TotalQuantity Controls =====")
print("Formula: AOV ~ rule_applied + BasketSize + AvgUnitPrice + TotalQuantity")
print(model_4_aov.summary())


# -------------------------
# Model 4B: log_AOV with BasketSize, AvgUnitPrice, TotalQuantity controls
# -------------------------
model_4_log = smf.ols(
    formula="log_AOV ~ rule_applied + BasketSize + AvgUnitPrice + TotalQuantity",
    data=reg_data
).fit(cov_type="HC3")

print("\n===== Model 4B: Log-Linear Regression with BasketSize, AvgUnitPrice, and TotalQuantity Controls =====")
print("Formula: log_AOV ~ rule_applied + BasketSize + AvgUnitPrice + TotalQuantity")
print(model_4_log.summary())


# -------------------------
# Build comparison table
# -------------------------
def extract_model_info(model, model_name):
    coef = model.params["rule_applied"]
    pval = model.pvalues["rule_applied"]
    ci = model.conf_int().loc["rule_applied"]

    return {
        "Model": model_name,
        "Rule_Coefficient": coef,
        "P_Value": pval,
        "CI_Lower": ci[0],
        "CI_Upper": ci[1],
        "R_Squared": model.rsquared,
        "Adj_R_Squared": model.rsquared_adj,
        "AIC": model.aic,
        "BIC": model.bic
    }

model_comparison_step8 = pd.DataFrame([
    extract_model_info(model_1_aov, "Model 1A - AOV ~ rule_applied"),
    extract_model_info(model_2_aov, "Model 2A - AOV ~ rule_applied + BasketSize"),
    extract_model_info(model_3_aov, "Model 3A - AOV ~ rule_applied + BasketSize + AvgUnitPrice"),
    extract_model_info(model_4_aov, "Model 4A - AOV ~ rule_applied + BasketSize + AvgUnitPrice + TotalQuantity"),
    extract_model_info(model_1_log, "Model 1B - log_AOV ~ rule_applied"),
    extract_model_info(model_2_log, "Model 2B - log_AOV ~ rule_applied + BasketSize"),
    extract_model_info(model_3_log, "Model 3B - log_AOV ~ rule_applied + BasketSize + AvgUnitPrice"),
    extract_model_info(model_4_log, "Model 4B - log_AOV ~ rule_applied + BasketSize + AvgUnitPrice + TotalQuantity")
])

print("\n===== Model Comparison Table =====")
display(model_comparison_step8.round(6))


# -------------------------
# Percentage interpretation for log models
# -------------------------
log_models = {
    "Model 1B baseline": model_1_log,
    "Model 2B + BasketSize": model_2_log,
    "Model 3B + BasketSize + AvgUnitPrice": model_3_log,
    "Model 4B + BasketSize + AvgUnitPrice + TotalQuantity": model_4_log
}

print("\n===== Percentage Effect Comparison from Log Models =====")
for name, model in log_models.items():
    coef = model.params["rule_applied"]
    pct_effect = (np.exp(coef) - 1) * 100
    pval = model.pvalues["rule_applied"]
    print(name)
    print("  coefficient:", round(coef, 6))
    print("  percentage effect:", round(pct_effect, 4), "%")
    print("  p-value:", round(pval, 6))


# -------------------------
# Coefficient change across models
# -------------------------
print("\n===== Change in rule_applied Coefficient Across Models =====")

print("AOV coefficient:")
print("Model 1A:", round(model_1_aov.params["rule_applied"], 4))
print("Model 2A:", round(model_2_aov.params["rule_applied"], 4))
print("Model 3A:", round(model_3_aov.params["rule_applied"], 4))
print("Model 4A:", round(model_4_aov.params["rule_applied"], 4))

print("\nlog_AOV coefficient:")
print("Model 1B:", round(model_1_log.params["rule_applied"], 6))
print("Model 2B:", round(model_2_log.params["rule_applied"], 6))
print("Model 3B:", round(model_3_log.params["rule_applied"], 6))
print("Model 4B:", round(model_4_log.params["rule_applied"], 6))


# -------------------------
# Save models for later steps
# -------------------------
models_step8 = {
    "model_1_aov": model_1_aov,
    "model_2_aov": model_2_aov,
    "model_3_aov": model_3_aov,
    "model_4_aov": model_4_aov,
    "model_1_log": model_1_log,
    "model_2_log": model_2_log,
    "model_3_log": model_3_log,
    "model_4_log": model_4_log
}

===== Regression Dataset Check =====
Shape: (38869, 18)
rule_applied counts:


,count
rule_applied,
0,38488
1,381



===== Correlation Matrix: Main Variables =====


,AOV,log_AOV,rule_applied,BasketSize,AvgUnitPrice,TotalQuantity
AOV,1.0000,0.6921,0.0572,0.4441,0.0092,0.7974
log_AOV,0.6921,1.0000,0.0594,0.4453,-0.0080,0.5957
rule_applied,0.0572,0.0594,1.0000,0.1176,-0.0242,0.0510
BasketSize,0.4441,0.4453,0.1176,1.0000,-0.0140,0.2899
AvgUnitPrice,0.0092,-0.0080,-0.0242,-0.0140,1.0000,-0.2370
TotalQuantity,0.7974,0.5957,0.0510,0.2899,-0.2370,1.0000



===== Group Comparison Before Step 8 Regression =====


,baskets,mean_AOV,median_AOV,mean_BasketSize,median_BasketSize,mean_AvgUnitPrice,median_AvgUnitPrice,mean_TotalQuantity,median_TotalQuantity
rule_applied,,,,,,,,,
0,38488,412.6647,298.8500,22.6192,15.0000,3.2624,2.8297,230.9834,146.0000
1,381,707.9793,459.5000,56.9606,44.0000,2.7885,2.5324,389.4252,267.0000



===== Model 4A: Linear Regression with BasketSize, AvgUnitPrice, and TotalQuantity Controls =====
Formula: AOV ~ rule_applied + BasketSize + AvgUnitPrice + TotalQuantity
                            OLS Regression Results                            
Dep. Variable:                    AOV   R-squared:                       0.722
Model:                            OLS   Adj. R-squared:                  0.722
Method:                 Least Squares   F-statistic:                     3042.
Date:                Thu, 04 Jun 2026   Prob (F-statistic):               0.00
Time:                        14:45:30   Log-Likelihood:            -2.7247e+05
No. Observations:               38869   AIC:                         5.450e+05
Df Residuals:                   38864   BIC:                         5.450e+05
Df Model:                           4                                         
Covariance Type:                  HC3                                         
                    coef    std err    

,Model,Rule_Coefficient,P_Value,CI_Lower,CI_Upper,R_Squared,Adj_R_Squared,AIC,BIC
0,Model 1A - AOV ~ rule_applied,295.3146,0.0000,222.6911,367.9381,0.0033,0.0032,594576.9682,594594.1041
1,Model 2A - AOV ~ rule_applied + BasketSize,26.3144,0.4200,-37.6436,90.2723,0.1972,0.1972,586167.1961,586192.9000
2,Model 3A - AOV ~ rule_applied + BasketSize + A...,28.1493,0.3878,-35.7302,92.0288,0.1975,0.1974,586157.4530,586191.7248
3,Model 4A - AOV ~ rule_applied + BasketSize + A...,-19.5486,0.3428,-59.9343,20.8371,0.7220,0.7220,544951.9042,544994.7440
4,Model 1B - log_AOV ~ rule_applied,0.7166,0.0000,0.6320,0.8013,0.0035,0.0035,123599.7111,123616.8470
5,Model 2B - log_AOV ~ rule_applied + BasketSize,0.0863,0.0429,0.0028,0.1698,0.1983,0.1983,115145.8524,115171.5563
6,Model 3B - log_AOV ~ rule_applied + BasketSize...,0.0858,0.0442,0.0022,0.1694,0.1984,0.1983,115147.7211,115181.9929
7,Model 4B - log_AOV ~ rule_applied + BasketSize...,0.0085,0.8154,-0.0628,0.0797,0.4505,0.4504,100470.2141,100513.0538



===== Percentage Effect Comparison from Log Models =====
Model 1B baseline
  coefficient: 0.716621
  percentage effect: 104.7503 %
  p-value: 0.0
Model 2B + BasketSize
  coefficient: 0.086263
  percentage effect: 9.0093 %
  p-value: 0.042916
Model 3B + BasketSize + AvgUnitPrice
  coefficient: 0.08581
  percentage effect: 8.9599 %
  p-value: 0.044237
Model 4B + BasketSize + AvgUnitPrice + TotalQuantity
  coefficient: 0.008485
  percentage effect: 0.8522 %
  p-value: 0.815404

===== Change in rule_applied Coefficient Across Models =====
AOV coefficient:
Model 1A: 295.3146
Model 2A: 26.3143
Model 3A: 28.1493
Model 4A: -19.5486

log_AOV coefficient:
Model 1B: 0.716621
Model 2B: 0.086263
Model 3B: 0.08581
Model 4B: 0.008485


# Step 9: Kiểm tra đa cộng tuyến bằng VIF

## Mục tiêu của bước này

Bước 9 nhằm kiểm tra xem các biến độc lập trong mô hình hồi quy có gặp vấn đề đa cộng tuyến hay không.

Ở Step 8, kết quả hồi quy có cảnh báo về condition number lớn. Cảnh báo này có thể xuất hiện khi các biến độc lập có tương quan cao với nhau hoặc khi các biến có thang đo rất khác nhau.

Vì vậy, trước khi thêm biến kiểm soát mới như `Country`, cần kiểm tra đa cộng tuyến bằng chỉ số VIF.

Các biến được kiểm tra gồm:

* `rule_applied`
* `BasketSize`
* `AvgUnitPrice`
* `TotalQuantity`

## Kết quả VIF

Kết quả VIF như sau:

| Variable      |    VIF |
| ------------- | -----: |
| rule_applied  | 1.0147 |
| BasketSize    | 1.1086 |
| AvgUnitPrice  | 1.0636 |
| TotalQuantity | 1.1607 |

Theo quy ước thông thường:

* VIF gần 1 cho thấy gần như không có đa cộng tuyến.
* VIF dưới 5 thường được xem là chấp nhận được.
* VIF trên 10 thường là dấu hiệu đa cộng tuyến nghiêm trọng.

Trong kết quả này, tất cả các biến đều có VIF rất thấp, xấp xỉ 1. Điều này cho thấy các biến độc lập trong mô hình không bị đa cộng tuyến nghiêm trọng.

## Kiểm tra tương quan giữa các biến độc lập

Correlation matrix giữa các biến độc lập cho thấy:

| Cặp biến                      | Correlation |
| ----------------------------- | ----------: |
| rule_applied và BasketSize    |      0.1176 |
| rule_applied và AvgUnitPrice  |     -0.0242 |
| rule_applied và TotalQuantity |      0.0510 |
| BasketSize và TotalQuantity   |      0.2899 |
| AvgUnitPrice và TotalQuantity |     -0.2370 |

Tương quan cao nhất là giữa `BasketSize` và `TotalQuantity`, với giá trị 0.2899. Đây là mức tương quan thấp đến trung bình, không đủ cao để gây lo ngại về đa cộng tuyến.

Điều này cho thấy `BasketSize` và `TotalQuantity` cùng đo lường quy mô basket, nhưng chúng vẫn phản ánh hai khía cạnh khác nhau:

* `BasketSize`: số lượng sản phẩm khác nhau trong basket.
* `TotalQuantity`: tổng số đơn vị sản phẩm được mua.

## Nhận xét về cảnh báo condition number

Mặc dù mô hình ở Step 8 có cảnh báo condition number lớn, kết quả VIF cho thấy vấn đề không phải là đa cộng tuyến nghiêm trọng.

Cảnh báo condition number nhiều khả năng đến từ việc các biến có thang đo khác nhau. Ví dụ, `rule_applied` là biến nhị phân 0/1, trong khi `TotalQuantity` có thể có giá trị lên tới vài nghìn.

Do đó, không cần loại bỏ biến nào khỏi mô hình tại thời điểm này.

## Kết luận bước 9

Kết quả VIF cho thấy các biến `rule_applied`, `BasketSize`, `AvgUnitPrice` và `TotalQuantity` đều có mức đa cộng tuyến rất thấp.

Vì vậy, bộ biến kiểm soát hiện tại có thể tiếp tục được sử dụng trong các mô hình tiếp theo. Bước tiếp theo có thể thêm biến `Country` vào mô hình để kiểm soát sự khác biệt về hành vi mua hàng giữa các quốc gia.


In [19]:
# =========================================
# STEP 9: MULTICOLLINEARITY CHECK USING VIF
# =========================================

import pandas as pd
import numpy as np
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm

# Use cleaned regression dataset
reg_data = model_reg_clean.copy()

print("===== Regression Dataset Check =====")
print("Shape:", reg_data.shape)

# -------------------------
# Select independent variables for VIF
# -------------------------
vif_vars = [
    "rule_applied",
    "BasketSize",
    "AvgUnitPrice",
    "TotalQuantity"
]

X_vif = reg_data[vif_vars].copy()

# Add constant for VIF calculation
X_vif_const = sm.add_constant(X_vif)

# -------------------------
# Calculate VIF
# -------------------------
vif_table = pd.DataFrame()
vif_table["Variable"] = X_vif_const.columns
vif_table["VIF"] = [
    variance_inflation_factor(X_vif_const.values, i)
    for i in range(X_vif_const.shape[1])
]

print("\n===== VIF Table =====")
display(vif_table.round(4))


# -------------------------
# Correlation matrix for reference
# -------------------------
print("\n===== Correlation Matrix among Independent Variables =====")
display(reg_data[vif_vars].corr().round(4))


# -------------------------
# Simple VIF interpretation
# -------------------------
print("\n===== VIF Interpretation =====")

for _, row in vif_table.iterrows():
    variable = row["Variable"]
    vif = row["VIF"]

    if variable == "const":
        continue

    if vif < 5:
        status = "Acceptable"
    elif vif < 10:
        status = "Potential concern"
    else:
        status = "Serious concern"

    print(f"{variable}: VIF = {vif:.4f} -> {status}")

===== Regression Dataset Check =====
Shape: (38869, 18)

===== VIF Table =====


,Variable,VIF
0,const,5.5070
1,rule_applied,1.0147
2,BasketSize,1.1086
3,AvgUnitPrice,1.0636
4,TotalQuantity,1.1607



===== Correlation Matrix among Independent Variables =====


,rule_applied,BasketSize,AvgUnitPrice,TotalQuantity
rule_applied,1.0000,0.1176,-0.0242,0.0510
BasketSize,0.1176,1.0000,-0.0140,0.2899
AvgUnitPrice,-0.0242,-0.0140,1.0000,-0.2370
TotalQuantity,0.0510,0.2899,-0.2370,1.0000



===== VIF Interpretation =====
rule_applied: VIF = 1.0147 -> Acceptable
BasketSize: VIF = 1.1086 -> Acceptable
AvgUnitPrice: VIF = 1.0636 -> Acceptable
TotalQuantity: VIF = 1.1607 -> Acceptable


# Step 10: Hồi quy với biến kiểm soát Country

## Mục tiêu của bước này

Bước 10 nhằm thêm biến `Country` vào mô hình hồi quy để kiểm soát sự khác biệt về hành vi mua hàng giữa các quốc gia.

Ở các bước trước, mô hình đã kiểm soát các yếu tố ở cấp độ basket, gồm:

* `BasketSize`: số lượng sản phẩm khác nhau trong basket.
* `AvgUnitPrice`: đơn giá trung bình của sản phẩm trong basket.
* `TotalQuantity`: tổng số lượng sản phẩm được mua.

Tuy nhiên, AOV cũng có thể khác nhau giữa các quốc gia do sự khác biệt về hành vi mua hàng, thị trường, mức giá, hoặc nhóm khách hàng. Vì vậy, bước này thêm `Country` vào mô hình dưới dạng biến phân loại.

Hai mô hình được chạy là:

```text
AOV ~ rule_applied + BasketSize + AvgUnitPrice + TotalQuantity + C(Country)
```

và:

```text
log_AOV ~ rule_applied + BasketSize + AvgUnitPrice + TotalQuantity + C(Country)
```

## Phân phối Country

Dữ liệu sau khi xử lý outlier có 43 quốc gia. Tuy nhiên, phân phối dữ liệu bị lệch mạnh về United Kingdom.

| Country        | Số basket |  Tỷ lệ |
| -------------- | --------: | -----: |
| United Kingdom |    35,625 | 91.65% |
| Germany        |       752 |  1.93% |
| France         |       595 |  1.53% |
| EIRE           |       562 |  1.45% |
| Netherlands    |       176 |  0.45% |

United Kingdom chiếm hơn 91% tổng số basket. Vì vậy, kết quả mô hình chủ yếu phản ánh hành vi mua hàng tại United Kingdom.

## Phân phối rule_applied theo Country

Rule được phân tích là:

```text
22748, 22745 → 22746
```

Trong dữ liệu sau cleaning, có 381 basket chứa đầy đủ rule. Trong đó, United Kingdom chiếm 328 basket.

Một số quốc gia khác có số lượng basket chứa rule khá nhỏ:

| Country        | Số basket có rule |
| -------------- | ----------------: |
| United Kingdom |               328 |
| France         |                15 |
| Germany        |                10 |
| EIRE           |                 6 |
| Spain          |                 5 |

Điều này cho thấy rule chủ yếu xuất hiện trong dữ liệu United Kingdom. Vì vậy, việc thêm Country control có thể giúp kiểm soát sự khác biệt giữa các nước, nhưng khó tạo ra thay đổi lớn cho hệ số của `rule_applied`.

## Kết quả mô hình AOV với Country control

Kết quả so sánh trước và sau khi thêm `Country` như sau:

| Model                      | Coefficient của rule_applied | p-value | R-squared |
| -------------------------- | ---------------------------: | ------: | --------: |
| Model 4A: không có Country |                     -19.5486 |  0.3428 |    0.7220 |
| Model 5A: có Country       |                     -20.1419 |  0.3277 |    0.7250 |

Sau khi thêm `Country`, hệ số của `rule_applied` thay đổi rất ít, từ -19.5486 thành -20.1419. p-value vẫn lớn hơn 0.05, cụ thể là 0.3277.

Điều này cho thấy trong mô hình AOV gốc, sau khi kiểm soát BasketSize, AvgUnitPrice, TotalQuantity và Country, chưa có bằng chứng thống kê cho thấy basket có chứa rule sẽ có AOV cao hơn.

R-squared tăng nhẹ từ 0.7220 lên 0.7250, cho thấy Country có bổ sung một phần thông tin cho mô hình, nhưng mức cải thiện không lớn.

## Kết quả mô hình log(AOV) với Country control

Ở mô hình log-linear, kết quả như sau:

| Model                      | Coefficient của rule_applied | Hiệu ứng phần trăm ước lượng | p-value |
| -------------------------- | ---------------------------: | ---------------------------: | ------: |
| Model 4B: không có Country |                       0.0085 |                        0.85% |  0.8154 |
| Model 5B: có Country       |                      -0.0035 |                       -0.35% |  0.9237 |

Sau khi thêm Country, hiệu ứng phần trăm của `rule_applied` chuyển từ khoảng 0.85% xuống còn -0.35%. p-value là 0.9237, rất lớn so với mức ý nghĩa 0.05.

Điều này cho thấy trong mô hình log(AOV) đầy đủ, `rule_applied` gần như không có liên hệ đáng kể với AOV.

## Nhận xét

Kết quả Step 10 xác nhận rằng việc thêm Country không làm thay đổi đáng kể kết luận từ Step 8.

Ở các mô hình đơn giản ban đầu, basket có rule nhìn có vẻ có AOV cao hơn. Tuy nhiên, sau khi lần lượt kiểm soát BasketSize, AvgUnitPrice, TotalQuantity và Country, hiệu ứng của `rule_applied` gần như biến mất.

Điều này cho thấy AOV cao hơn ở nhóm basket có rule chủ yếu được giải thích bởi quy mô giỏ hàng và tổng số lượng sản phẩm mua, thay vì bản thân association rule.

## Lưu ý về mô hình

Kết quả mô hình có cảnh báo condition number lớn. Một phần nguyên nhân có thể đến từ việc thêm nhiều biến giả cho Country, trong khi một số quốc gia có số lượng quan sát rất nhỏ. Ngoài ra, các biến numeric cũng có thang đo khác nhau.

Tuy nhiên, ở Step 9, kiểm tra VIF cho các biến numeric chính cho thấy không có đa cộng tuyến nghiêm trọng. Vì vậy, cảnh báo này cần được ghi nhận nhưng không làm thay đổi kết luận chính về `rule_applied`.

## Kết luận bước 10

Sau khi thêm Country vào mô hình, `rule_applied` vẫn không có ý nghĩa thống kê trong cả mô hình AOV và log(AOV).

Kết quả cuối cùng cho thấy chưa có bằng chứng mạnh rằng rule `22748, 22745 → 22746` tạo ra tác động độc lập làm tăng AOV sau khi đã kiểm soát các yếu tố quan trọng như quy mô giỏ hàng, đơn giá trung bình, tổng số lượng mua và quốc gia.

Do đó, association rule này có thể hữu ích cho việc nhận diện các sản phẩm thường được mua cùng nhau, nhưng chưa đủ bằng chứng để khẳng định rule này trực tiếp làm tăng AOV.


In [22]:
# ===============================================================
# STEP 10: REGRESSION WITH COUNTRY CONTROL
# AOV ~ rule_applied + BasketSize + AvgUnitPrice + TotalQuantity + C(Country)
# log_AOV ~ rule_applied + BasketSize + AvgUnitPrice + TotalQuantity + C(Country)
# ===============================================================

import statsmodels.formula.api as smf
import numpy as np
import pandas as pd

# Use cleaned regression dataset
reg_data = model_reg_clean.copy()

print("===== Regression Dataset Check =====")
print("Shape:", reg_data.shape)
print("rule_applied counts:")
display(reg_data["rule_applied"].value_counts().sort_index().to_frame("count"))

print("\n===== Country Distribution =====")
country_counts = reg_data["Country"].value_counts().to_frame("count")
country_counts["rate"] = country_counts["count"] / len(reg_data)
display(country_counts.head(20).round(4))

print("\nNumber of unique countries:", reg_data["Country"].nunique())


# -------------------------
# Check rule_applied by country
# -------------------------
print("\n===== rule_applied by Country =====")
rule_by_country = reg_data.groupby("Country").agg(
    baskets=("InvoiceNo", "count"),
    rule_applied_baskets=("rule_applied", "sum"),
    rule_applied_rate=("rule_applied", "mean"),
    mean_AOV=("AOV", "mean"),
    median_AOV=("AOV", "median")
).sort_values("baskets", ascending=False).round(4)

display(rule_by_country.head(20))


# -------------------------
# Refit previous Model 4 for comparison
# -------------------------
model_4_aov = smf.ols(
    formula="AOV ~ rule_applied + BasketSize + AvgUnitPrice + TotalQuantity",
    data=reg_data
).fit(cov_type="HC3")

model_4_log = smf.ols(
    formula="log_AOV ~ rule_applied + BasketSize + AvgUnitPrice + TotalQuantity",
    data=reg_data
).fit(cov_type="HC3")


# -------------------------
# Model 5A: AOV with Country control
# -------------------------
model_5_aov = smf.ols(
    formula="AOV ~ rule_applied + BasketSize + AvgUnitPrice + TotalQuantity + C(Country)",
    data=reg_data
).fit(cov_type="HC3")

print("\n===== Model 5A: Linear Regression with Country Control =====")
print("Formula: AOV ~ rule_applied + BasketSize + AvgUnitPrice + TotalQuantity + C(Country)")
print(model_5_aov.summary())


# -------------------------
# Model 5B: log_AOV with Country control
# -------------------------
model_5_log = smf.ols(
    formula="log_AOV ~ rule_applied + BasketSize + AvgUnitPrice + TotalQuantity + C(Country)",
    data=reg_data
).fit(cov_type="HC3")

print("\n===== Model 5B: Log-Linear Regression with Country Control =====")
print("Formula: log_AOV ~ rule_applied + BasketSize + AvgUnitPrice + TotalQuantity + C(Country)")
print(model_5_log.summary())


# -------------------------
# Build comparison table
# -------------------------
def extract_model_info(model, model_name):
    coef = model.params["rule_applied"]
    pval = model.pvalues["rule_applied"]
    ci = model.conf_int().loc["rule_applied"]

    return {
        "Model": model_name,
        "Rule_Coefficient": coef,
        "P_Value": pval,
        "CI_Lower": ci[0],
        "CI_Upper": ci[1],
        "R_Squared": model.rsquared,
        "Adj_R_Squared": model.rsquared_adj,
        "AIC": model.aic,
        "BIC": model.bic
    }

model_comparison_step10 = pd.DataFrame([
    extract_model_info(model_4_aov, "Model 4A - AOV ~ rule + BasketSize + AvgUnitPrice + TotalQuantity"),
    extract_model_info(model_5_aov, "Model 5A - AOV ~ rule + BasketSize + AvgUnitPrice + TotalQuantity + Country"),
    extract_model_info(model_4_log, "Model 4B - log_AOV ~ rule + BasketSize + AvgUnitPrice + TotalQuantity"),
    extract_model_info(model_5_log, "Model 5B - log_AOV ~ rule + BasketSize + AvgUnitPrice + TotalQuantity + Country")
])

print("\n===== Model Comparison Table: Before vs After Country Control =====")
display(model_comparison_step10.round(6))


# -------------------------
# Percentage interpretation for log models
# -------------------------
print("\n===== Percentage Effect Comparison from Log Models =====")

coef_4_log = model_4_log.params["rule_applied"]
coef_5_log = model_5_log.params["rule_applied"]

pct_4_log = (np.exp(coef_4_log) - 1) * 100
pct_5_log = (np.exp(coef_5_log) - 1) * 100

print("Model 4B without Country:")
print("  coefficient:", round(coef_4_log, 6))
print("  percentage effect:", round(pct_4_log, 4), "%")
print("  p-value:", round(model_4_log.pvalues["rule_applied"], 6))

print("\nModel 5B with Country:")
print("  coefficient:", round(coef_5_log, 6))
print("  percentage effect:", round(pct_5_log, 4), "%")
print("  p-value:", round(model_5_log.pvalues["rule_applied"], 6))


# -------------------------
# Coefficient change after adding Country
# -------------------------
print("\n===== Change in rule_applied Coefficient After Adding Country =====")

print("AOV coefficient:")
print("Model 4A before Country:", round(model_4_aov.params["rule_applied"], 4))
print("Model 5A after Country:", round(model_5_aov.params["rule_applied"], 4))
print("Change:", round(model_5_aov.params["rule_applied"] - model_4_aov.params["rule_applied"], 4))

print("\nlog_AOV coefficient:")
print("Model 4B before Country:", round(model_4_log.params["rule_applied"], 6))
print("Model 5B after Country:", round(model_5_log.params["rule_applied"], 6))
print("Change:", round(model_5_log.params["rule_applied"] - model_4_log.params["rule_applied"], 6))


# -------------------------
# Save models for later steps
# -------------------------
models_step10 = {
    "model_4_aov": model_4_aov,
    "model_5_aov": model_5_aov,
    "model_4_log": model_4_log,
    "model_5_log": model_5_log
}

===== Regression Dataset Check =====
Shape: (38869, 18)
rule_applied counts:


,count
rule_applied,
0,38488
1,381



===== Country Distribution =====


,count,rate
Country,,
United Kingdom,35625,0.9165
Germany,752,0.0193
France,595,0.0153
EIRE,562,0.0145
Netherlands,176,0.0045
Spain,144,0.0037
Belgium,143,0.0037
Sweden,93,0.0024
Portugal,84,0.0022



Number of unique countries: 43

===== rule_applied by Country =====


,baskets,rule_applied_baskets,rule_applied_rate,mean_AOV,median_AOV
Country,,,,,
United Kingdom,35625,328,0.0092,396.6250,292.3400
Germany,752,10,0.0133,497.2709,360.1500
France,595,15,0.0252,477.6207,358.8800
EIRE,562,6,0.0107,789.9265,662.4550
Netherlands,176,1,0.0057,945.5701,418.0000
Spain,144,5,0.0347,678.9358,364.3700
Belgium,143,3,0.0210,398.5536,301.5600
Sweden,93,1,0.0108,662.9588,347.5600
Portugal,84,1,0.0119,561.7681,363.5200



===== Model 5A: Linear Regression with Country Control =====
Formula: AOV ~ rule_applied + BasketSize + AvgUnitPrice + TotalQuantity + C(Country)
                            OLS Regression Results                            
Dep. Variable:                    AOV   R-squared:                       0.725
Model:                            OLS   Adj. R-squared:                  0.725
Method:                 Least Squares   F-statistic:                     324.7
Date:                Thu, 04 Jun 2026   Prob (F-statistic):               0.00
Time:                        14:50:46   Log-Likelihood:            -2.7226e+05
No. Observations:               38869   AIC:                         5.446e+05
Df Residuals:                   38822   BIC:                         5.450e+05
Df Model:                          46                                         
Covariance Type:                  HC3                                         
                                         coef    std err       

,Model,Rule_Coefficient,P_Value,CI_Lower,CI_Upper,R_Squared,Adj_R_Squared,AIC,BIC
0,Model 4A - AOV ~ rule + BasketSize + AvgUnitPr...,-19.5486,0.3428,-59.9343,20.8371,0.7220,0.7220,544951.9042,544994.7440
1,Model 5A - AOV ~ rule + BasketSize + AvgUnitPr...,-20.1419,0.3277,-60.4795,20.1957,0.7250,0.7247,544610.5849,545013.2787
2,Model 4B - log_AOV ~ rule + BasketSize + AvgUn...,0.0085,0.8154,-0.0628,0.0797,0.4505,0.4504,100470.2141,100513.0538
3,Model 5B - log_AOV ~ rule + BasketSize + AvgUn...,-0.0035,0.9237,-0.0744,0.0674,0.4552,0.4545,100220.9736,100623.6674



===== Percentage Effect Comparison from Log Models =====
Model 4B without Country:
  coefficient: 0.008485
  percentage effect: 0.8522 %
  p-value: 0.815404

Model 5B with Country:
  coefficient: -0.003465
  percentage effect: -0.3459 %
  p-value: 0.92369

===== Change in rule_applied Coefficient After Adding Country =====
AOV coefficient:
Model 4A before Country: -19.5486
Model 5A after Country: -20.1419
Change: -0.5933

log_AOV coefficient:
Model 4B before Country: 0.008485
Model 5B after Country: -0.003465
Change: -0.01195


# Step 11: Quantify Incremental Sales

## Objective

The goal of this step is to estimate the incremental sales associated with the selected association rule.

The selected rule is:

`22748, 22745 → 22746`

In the previous regression steps, the baseline model showed a positive relationship between `rule_applied` and AOV. However, after controlling for BasketSize, AvgUnitPrice, TotalQuantity, and Country, the coefficient of `rule_applied` became statistically insignificant.

Therefore, incremental sales are calculated in two ways:

1. Naive estimate:
   - Based on the baseline model without controls.
   - This shows the raw AOV difference between baskets with and without the rule.

2. Controlled estimate:
   - Based on the final regression model with controls.# Step 11: Quantify Incremental Sales

## Mục tiêu của bước này

Bước 11 nhằm ước lượng incremental sales liên quan đến association rule đã chọn:

```text
22748, 22745 → 22746
```

Incremental sales được hiểu là phần doanh thu tăng thêm ước lượng khi basket có chứa rule hoặc khi rule được dùng để gợi ý cross-sell.

Tuy nhiên, ở các bước hồi quy trước đó, kết quả cho thấy hệ số của `rule_applied` chỉ dương và có ý nghĩa trong mô hình baseline. Sau khi thêm các biến kiểm soát như `BasketSize`, `AvgUnitPrice`, `TotalQuantity` và `Country`, hệ số của `rule_applied` không còn có ý nghĩa thống kê.

Vì vậy, incremental sales được tính theo hai cách:

1. **Naive estimate**: dùng hệ số từ mô hình baseline, chưa kiểm soát các yếu tố gây nhiễu.
2. **Controlled estimate**: dùng hệ số từ mô hình đầy đủ, đã kiểm soát các yếu tố quan trọng.

## Số lượng basket được sử dụng

Sau khi xử lý outlier, dữ liệu có:

| Loại basket                      | Số lượng |
| -------------------------------- | -------: |
| Observed rule-applied baskets    |      381 |
| Recommendation candidate baskets |      132 |

Trong đó:

* `Observed rule-applied baskets` là các basket đã chứa đầy đủ cả vế trái và vế phải của rule.
* `Recommendation candidate baskets` là các basket có chứa vế trái của rule nhưng chưa chứa vế phải, tức là nhóm có thể được dùng để mô phỏng cross-sell.

Ở bước kiểm tra ban đầu, số lượng recommendation candidates là 148. Sau khi xử lý outlier, con số này còn 132, vì một số basket cực đoan đã bị loại khỏi dữ liệu regression sạch.

## Naive incremental sales estimate

Naive estimate sử dụng hệ số từ mô hình baseline:

```text
AOV ~ rule_applied
```

Hệ số của `rule_applied` trong mô hình baseline là:

```text
295.3146
```

Kết quả incremental sales naive như sau:

| Estimate type             | Coefficient used | Number of baskets | Incremental sales estimate | p-value |
| ------------------------- | ---------------: | ----------------: | -------------------------: | ------: |
| Observed rule baskets     |         295.3146 |               381 |                 112,514.85 |  0.0000 |
| Recommendation candidates |         295.3146 |               132 |                  38,981.52 |  0.0000 |

Theo naive estimate, các basket có rule có vẻ liên quan đến mức doanh thu cao hơn. Nếu chỉ nhìn mô hình baseline, observed rule baskets có incremental sales ước lượng khoảng 112,514.85, còn nhóm recommendation candidates có incremental sales tiềm năng khoảng 38,981.52.

Tuy nhiên, đây chỉ là kết quả thô vì mô hình baseline chưa kiểm soát các yếu tố gây nhiễu. Do đó, không nên dùng con số này để kết luận rule thật sự tạo ra doanh thu tăng thêm.

## Controlled incremental sales estimate

Controlled estimate sử dụng hệ số từ mô hình đầy đủ:

```text
AOV ~ rule_applied + BasketSize + AvgUnitPrice + TotalQuantity + C(Country)
```

Hệ số của `rule_applied` trong mô hình đầy đủ là:

```text
-20.1419
```

Kết quả controlled incremental sales như sau:

| Estimate type             | Coefficient used | Number of baskets | Incremental sales estimate | p-value |
| ------------------------- | ---------------: | ----------------: | -------------------------: | ------: |
| Observed rule baskets     |         -20.1419 |               381 |                  -7,674.07 |  0.3277 |
| Recommendation candidates |         -20.1419 |               132 |                  -2,658.73 |  0.3277 |

Sau khi kiểm soát BasketSize, AvgUnitPrice, TotalQuantity và Country, hệ số của `rule_applied` không còn dương. Tuy nhiên, p-value là 0.3277, lớn hơn mức ý nghĩa 0.05.

Vì vậy, không thể kết luận rằng rule làm giảm doanh thu. Cách diễn giải đúng là chưa có bằng chứng thống kê đủ mạnh để khẳng định rule tạo ra incremental sales dương sau khi đã kiểm soát các yếu tố quan trọng.

## Kết quả từ mô hình log(AOV)

Kết quả mô hình log(AOV) cũng cho thấy sự khác biệt rõ giữa baseline và final controlled model:

| Model                      | Coefficient của rule_applied | Hiệu ứng phần trăm ước lượng | p-value |
| -------------------------- | ---------------------------: | ---------------------------: | ------: |
| Baseline log model         |                       0.7166 |                      104.75% |  0.0000 |
| Final controlled log model |                      -0.0035 |                       -0.35% |  0.9237 |

Ở baseline log model, basket có rule có AOV cao hơn khoảng 104.75%. Tuy nhiên, sau khi thêm các biến kiểm soát, hiệu ứng chỉ còn khoảng -0.35% và hoàn toàn không có ý nghĩa thống kê.

Điều này xác nhận rằng mối quan hệ dương ban đầu giữa rule và AOV chủ yếu đến từ các yếu tố khác như quy mô basket và tổng số lượng sản phẩm mua.

## Nhận xét

Kết quả Step 11 cho thấy nếu chỉ dùng mô hình baseline, rule có vẻ tạo ra incremental sales lớn. Tuy nhiên, sau khi kiểm soát các yếu tố quan trọng, incremental sales không còn dương và không có ý nghĩa thống kê.

Do đó, không thể khẳng định rule `22748, 22745 → 22746` tạo ra doanh thu tăng thêm độc lập.

Rule này vẫn có giá trị trong Market Basket Analysis vì nó cho thấy ba sản phẩm có xu hướng được mua cùng nhau rất mạnh. Tuy nhiên, xét từ góc độ tăng AOV, kết quả hồi quy chưa cung cấp bằng chứng đủ mạnh rằng rule này trực tiếp làm tăng doanh thu.

## Kết luận bước 11

Naive estimate cho thấy rule có liên quan đến AOV cao hơn, nhưng controlled estimate cho thấy không có bằng chứng thống kê về incremental sales dương sau khi kiểm soát BasketSize, AvgUnitPrice, TotalQuantity và Country.

Vì vậy, kết luận phù hợp là:

```text
The selected association rule identifies products that are frequently bought together, but there is no statistically significant evidence that the rule independently generates positive incremental sales after controlling for basket size, price, quantity, and country.
```

Dịch sang tiếng Việt:

```text
Rule được chọn giúp nhận diện các sản phẩm thường được mua cùng nhau, nhưng chưa có bằng chứng thống kê đủ mạnh cho thấy rule này tạo ra incremental sales dương một cách độc lập sau khi kiểm soát quy mô giỏ hàng, giá trung bình, tổng số lượng mua và quốc gia.
```

   - This is more appropriate for business interpretation.

## Important Interpretation

Because the final controlled coefficient is not statistically significant, the controlled incremental sales estimate should not be interpreted as strong evidence of actual sales uplift.

Instead, the result should be interpreted as:

The selected rule identifies products that are frequently bought together, but there is not enough statistical evidence that the rule independently increases AOV after controlling for basket size, price, quantity, and country.

## Input

- `model_reg_clean`

## Output

- Number of observed rule-applied baskets
- Number of recommendation candidates
- Naive incremental sales estimate
- Controlled incremental sales estimate
- Final interpretation

In [23]:
# =========================================
# STEP 11: QUANTIFY INCREMENTAL SALES
# =========================================

import statsmodels.formula.api as smf
import numpy as np
import pandas as pd

# Use cleaned regression dataset
reg_data = model_reg_clean.copy()

print("===== Dataset Check =====")
print("Shape:", reg_data.shape)

# -------------------------
# Count observed rule baskets and recommendation candidates
# -------------------------
observed_rule_baskets = reg_data["rule_applied"].sum()
recommendation_candidates = reg_data["recommendation_candidate"].sum()

print("\n===== Rule Basket Counts =====")
print("Observed rule-applied baskets:", observed_rule_baskets)
print("Recommendation candidate baskets:", recommendation_candidates)


# -------------------------
# Refit key models
# -------------------------
model_baseline_aov = smf.ols(
    formula="AOV ~ rule_applied",
    data=reg_data
).fit(cov_type="HC3")

model_final_aov = smf.ols(
    formula="AOV ~ rule_applied + BasketSize + AvgUnitPrice + TotalQuantity + C(Country)",
    data=reg_data
).fit(cov_type="HC3")

model_baseline_log = smf.ols(
    formula="log_AOV ~ rule_applied",
    data=reg_data
).fit(cov_type="HC3")

model_final_log = smf.ols(
    formula="log_AOV ~ rule_applied + BasketSize + AvgUnitPrice + TotalQuantity + C(Country)",
    data=reg_data
).fit(cov_type="HC3")


# -------------------------
# Extract coefficients
# -------------------------
baseline_aov_coef = model_baseline_aov.params["rule_applied"]
baseline_aov_pval = model_baseline_aov.pvalues["rule_applied"]

final_aov_coef = model_final_aov.params["rule_applied"]
final_aov_pval = model_final_aov.pvalues["rule_applied"]

baseline_log_coef = model_baseline_log.params["rule_applied"]
baseline_log_pval = model_baseline_log.pvalues["rule_applied"]
baseline_pct_effect = (np.exp(baseline_log_coef) - 1) * 100

final_log_coef = model_final_log.params["rule_applied"]
final_log_pval = model_final_log.pvalues["rule_applied"]
final_pct_effect = (np.exp(final_log_coef) - 1) * 100


# -------------------------
# Incremental sales estimates
# -------------------------
# 1. Observed historical rule baskets
naive_incremental_sales_observed = baseline_aov_coef * observed_rule_baskets
controlled_incremental_sales_observed = final_aov_coef * observed_rule_baskets

# 2. Recommendation candidate baskets
# This estimates potential impact if the same coefficient applied to candidate baskets.
naive_incremental_sales_candidates = baseline_aov_coef * recommendation_candidates
controlled_incremental_sales_candidates = final_aov_coef * recommendation_candidates


# -------------------------
# Summary table
# -------------------------
incremental_summary = pd.DataFrame({
    "Estimate_Type": [
        "Naive estimate - observed rule baskets",
        "Controlled estimate - observed rule baskets",
        "Naive estimate - recommendation candidates",
        "Controlled estimate - recommendation candidates"
    ],
    "Coefficient_Used": [
        baseline_aov_coef,
        final_aov_coef,
        baseline_aov_coef,
        final_aov_coef
    ],
    "Number_of_Baskets": [
        observed_rule_baskets,
        observed_rule_baskets,
        recommendation_candidates,
        recommendation_candidates
    ],
    "Incremental_Sales_Estimate": [
        naive_incremental_sales_observed,
        controlled_incremental_sales_observed,
        naive_incremental_sales_candidates,
        controlled_incremental_sales_candidates
    ],
    "P_Value_of_Coefficient": [
        baseline_aov_pval,
        final_aov_pval,
        baseline_aov_pval,
        final_aov_pval
    ]
})

print("\n===== Incremental Sales Summary =====")
display(incremental_summary.round(4))


# -------------------------
# Log model interpretation table
# -------------------------
log_effect_summary = pd.DataFrame({
    "Model": [
        "Baseline log model",
        "Final controlled log model"
    ],
    "Rule_Coefficient": [
        baseline_log_coef,
        final_log_coef
    ],
    "Approx_Percentage_Effect": [
        baseline_pct_effect,
        final_pct_effect
    ],
    "P_Value": [
        baseline_log_pval,
        final_log_pval
    ]
})

print("\n===== Log Model Percentage Effect Summary =====")
display(log_effect_summary.round(4))


# -------------------------
# Final interpretation helper
# -------------------------
print("\n===== Final Interpretation Helper =====")
print("Baseline AOV coefficient:", round(baseline_aov_coef, 4))
print("Baseline p-value:", round(baseline_aov_pval, 6))
print("Naive incremental sales for observed rule baskets:", round(naive_incremental_sales_observed, 2))

print("\nFinal controlled AOV coefficient:", round(final_aov_coef, 4))
print("Final controlled p-value:", round(final_aov_pval, 6))
print("Controlled incremental sales for observed rule baskets:", round(controlled_incremental_sales_observed, 2))

print("\nBaseline log percentage effect:", round(baseline_pct_effect, 4), "%")
print("Final controlled log percentage effect:", round(final_pct_effect, 4), "%")

if final_aov_pval < 0.05 and final_aov_coef > 0:
    print("\nConclusion: There is statistically significant evidence of positive incremental sales after controls.")
else:
    print("\nConclusion: There is no statistically significant evidence of positive incremental sales after controls.")

===== Dataset Check =====
Shape: (38869, 18)

===== Rule Basket Counts =====
Observed rule-applied baskets: 381
Recommendation candidate baskets: 132

===== Incremental Sales Summary =====


,Estimate_Type,Coefficient_Used,Number_of_Baskets,Incremental_Sales_Estimate,P_Value_of_Coefficient
0,Naive estimate - observed rule baskets,295.3146,381,112514.8498,0.0000
1,Controlled estimate - observed rule baskets,-20.1419,381,-7674.0683,0.3277
2,Naive estimate - recommendation candidates,295.3146,132,38981.5228,0.0000
3,Controlled estimate - recommendation candidates,-20.1419,132,-2658.7323,0.3277



===== Log Model Percentage Effect Summary =====


,Model,Rule_Coefficient,Approx_Percentage_Effect,P_Value
0,Baseline log model,0.7166,104.7503,0.0000
1,Final controlled log model,-0.0035,-0.3459,0.9237



===== Final Interpretation Helper =====
Baseline AOV coefficient: 295.3146
Baseline p-value: 0.0
Naive incremental sales for observed rule baskets: 112514.85

Final controlled AOV coefficient: -20.1419
Final controlled p-value: 0.32774
Controlled incremental sales for observed rule baskets: -7674.07

Baseline log percentage effect: 104.7503 %
Final controlled log percentage effect: -0.3459 %

Conclusion: There is no statistically significant evidence of positive incremental sales after controls.


# Step 12: Causal Impact Estimation

## Mục tiêu của bước này

Bước 12 nhằm tổng hợp kết quả hồi quy để đánh giá liệu association rule được chọn có thể được xem là có tác động nhân quả lên AOV hay không.

Rule được phân tích là:

```text
22748, 22745 → 22746
```

Ở các bước trước, biến `rule_applied` được tạo để xác định basket có chứa đầy đủ rule hay không. Sau đó, các mô hình hồi quy được xây dựng theo từng cấp độ kiểm soát biến để kiểm tra xem mối liên hệ giữa rule và AOV có còn tồn tại sau khi thêm các yếu tố gây nhiễu hay không.

## Kết quả tổng hợp các mô hình

Kết quả cho thấy ở mô hình baseline, `rule_applied` có hệ số dương và có ý nghĩa thống kê.

Cụ thể:

| Model            | Coefficient của rule_applied | p-value | Kết luận            |
| ---------------- | ---------------------------: | ------: | ------------------- |
| Baseline AOV     |                     295.3146 |  0.0000 | Dương và có ý nghĩa |
| Baseline log_AOV |                       0.7166 |  0.0000 | Dương và có ý nghĩa |

Ở mô hình baseline, basket có rule có AOV cao hơn khoảng 295.31 đơn vị so với basket không có rule. Trong mô hình log(AOV), basket có rule có AOV cao hơn khoảng 104.75%.

Tuy nhiên, đây chỉ là mối quan hệ thô vì mô hình baseline chưa kiểm soát các yếu tố khác như quy mô giỏ hàng, tổng số lượng sản phẩm, giá trung bình và quốc gia.

## Kết quả sau khi thêm biến kiểm soát

Khi lần lượt thêm các biến kiểm soát, hệ số của `rule_applied` giảm mạnh.

Ở mô hình cuối cùng:

```text
AOV ~ rule_applied + BasketSize + AvgUnitPrice + TotalQuantity + C(Country)
```

kết quả là:

| Chỉ số                       |  Giá trị |
| ---------------------------- | -------: |
| Coefficient của rule_applied | -20.1419 |
| p-value                      |   0.3277 |

Ở mô hình log(AOV) cuối cùng:

```text
log_AOV ~ rule_applied + BasketSize + AvgUnitPrice + TotalQuantity + C(Country)
```

kết quả là:

| Chỉ số                       |  Giá trị |
| ---------------------------- | -------: |
| Coefficient của rule_applied |  -0.0035 |
| Hiệu ứng phần trăm ước lượng | -0.3459% |
| p-value                      |   0.9237 |

Cả hai mô hình cuối cùng đều cho thấy `rule_applied` không có ý nghĩa thống kê.

## Diễn giải causal impact

Kết quả này cho thấy mối quan hệ dương giữa rule và AOV ở mô hình baseline chủ yếu bị ảnh hưởng bởi các yếu tố gây nhiễu, đặc biệt là quy mô basket và tổng số lượng sản phẩm mua.

Nói cách khác, các basket có rule thường là các basket lớn hơn hoặc có tổng số lượng sản phẩm cao hơn. Vì vậy, AOV cao hơn ở nhóm có rule không nhất thiết đến từ bản thân rule.

Sau khi kiểm soát các yếu tố quan trọng như `BasketSize`, `AvgUnitPrice`, `TotalQuantity` và `Country`, không còn bằng chứng thống kê cho thấy rule tạo ra tác động độc lập làm tăng AOV.

## Giới hạn về dữ liệu

Dữ liệu được sử dụng là dữ liệu giao dịch lịch sử, không phải dữ liệu từ A/B testing. Do đó, mô hình hồi quy không thể chứng minh quan hệ nhân quả tuyệt đối.

Regression trong trường hợp này chỉ giúp ước lượng mối liên hệ giữa `rule_applied` và AOV sau khi kiểm soát các biến quan sát được.

Vì không có thiết kế thí nghiệm ngẫu nhiên, không nên kết luận rằng rule chắc chắn gây ra thay đổi trong AOV.

## Kết luận bước 12

Kết luận cuối cùng là:

```text
No statistically significant evidence of positive causal impact after controls.
```

Dịch sang tiếng Việt:

```text
Sau khi kiểm soát các yếu tố quan trọng, không có bằng chứng thống kê đủ mạnh cho thấy association rule có tác động nhân quả dương lên AOV.
```

Rule `22748, 22745 → 22746` vẫn có giá trị trong Market Basket Analysis vì nó cho thấy các sản phẩm này thường được mua cùng nhau. Tuy nhiên, xét từ góc độ causal impact và incremental sales, chưa có bằng chứng mạnh cho thấy rule này trực tiếp làm tăng AOV sau khi kiểm soát quy mô giỏ hàng, giá trung bình, tổng số lượng mua và quốc gia.


In [25]:
# =========================================
# STEP 12: CAUSAL IMPACT ESTIMATION SUMMARY
# =========================================

import statsmodels.formula.api as smf
import numpy as np
import pandas as pd

# Use cleaned regression dataset
reg_data = model_reg_clean.copy()

# -------------------------
# Refit all key models for final causal summary
# -------------------------
models = {
    "Model 1A - Baseline AOV": smf.ols(
        "AOV ~ rule_applied", data=reg_data
    ).fit(cov_type="HC3"),

    "Model 2A - + BasketSize": smf.ols(
        "AOV ~ rule_applied + BasketSize", data=reg_data
    ).fit(cov_type="HC3"),

    "Model 3A - + BasketSize + AvgUnitPrice": smf.ols(
        "AOV ~ rule_applied + BasketSize + AvgUnitPrice", data=reg_data
    ).fit(cov_type="HC3"),

    "Model 4A - + BasketSize + AvgUnitPrice + TotalQuantity": smf.ols(
        "AOV ~ rule_applied + BasketSize + AvgUnitPrice + TotalQuantity", data=reg_data
    ).fit(cov_type="HC3"),

    "Model 5A - + BasketSize + AvgUnitPrice + TotalQuantity + Country": smf.ols(
        "AOV ~ rule_applied + BasketSize + AvgUnitPrice + TotalQuantity + C(Country)", data=reg_data
    ).fit(cov_type="HC3"),

    "Model 1B - Baseline log_AOV": smf.ols(
        "log_AOV ~ rule_applied", data=reg_data
    ).fit(cov_type="HC3"),

    "Model 2B - log_AOV + BasketSize": smf.ols(
        "log_AOV ~ rule_applied + BasketSize", data=reg_data
    ).fit(cov_type="HC3"),

    "Model 3B - log_AOV + BasketSize + AvgUnitPrice": smf.ols(
        "log_AOV ~ rule_applied + BasketSize + AvgUnitPrice", data=reg_data
    ).fit(cov_type="HC3"),

    "Model 4B - log_AOV + BasketSize + AvgUnitPrice + TotalQuantity": smf.ols(
        "log_AOV ~ rule_applied + BasketSize + AvgUnitPrice + TotalQuantity", data=reg_data
    ).fit(cov_type="HC3"),

    "Model 5B - log_AOV + BasketSize + AvgUnitPrice + TotalQuantity + Country": smf.ols(
        "log_AOV ~ rule_applied + BasketSize + AvgUnitPrice + TotalQuantity + C(Country)", data=reg_data
    ).fit(cov_type="HC3")
}


# -------------------------
# Build causal summary table
# -------------------------
rows = []

for model_name, model in models.items():
    coef = model.params["rule_applied"]
    pval = model.pvalues["rule_applied"]
    ci = model.conf_int().loc["rule_applied"]

    if "log_AOV" in model_name:
        pct_effect = (np.exp(coef) - 1) * 100
    else:
        pct_effect = np.nan

    if pval < 0.05 and coef > 0:
        interpretation = "Positive and statistically significant"
    elif pval < 0.05 and coef < 0:
        interpretation = "Negative and statistically significant"
    else:
        interpretation = "Not statistically significant"

    rows.append({
        "Model": model_name,
        "Rule_Coefficient": coef,
        "P_Value": pval,
        "CI_Lower": ci[0],
        "CI_Upper": ci[1],
        "R_Squared": model.rsquared,
        "Adj_R_Squared": model.rsquared_adj,
        "Approx_Percentage_Effect_Log_Models": pct_effect,
        "Interpretation": interpretation
    })

causal_summary = pd.DataFrame(rows)

print("===== Final Causal Impact Summary Table =====")
display(causal_summary.round(6))


# -------------------------
# Final causal decision
# -------------------------
final_aov_model = models["Model 5A - + BasketSize + AvgUnitPrice + TotalQuantity + Country"]
final_log_model = models["Model 5B - log_AOV + BasketSize + AvgUnitPrice + TotalQuantity + Country"]

final_aov_coef = final_aov_model.params["rule_applied"]
final_aov_pval = final_aov_model.pvalues["rule_applied"]

final_log_coef = final_log_model.params["rule_applied"]
final_log_pval = final_log_model.pvalues["rule_applied"]
final_log_pct = (np.exp(final_log_coef) - 1) * 100

print("\n===== Final Causal Interpretation =====")
print("Final AOV model coefficient:", round(final_aov_coef, 4))
print("Final AOV model p-value:", round(final_aov_pval, 6))

print("\nFinal log_AOV model coefficient:", round(final_log_coef, 6))
print("Final log_AOV percentage effect:", round(final_log_pct, 4), "%")
print("Final log_AOV model p-value:", round(final_log_pval, 6))

if final_aov_pval < 0.05 and final_aov_coef > 0:
    causal_conclusion = "Potential positive causal impact, but still limited because the data is observational."
else:
    causal_conclusion = "No statistically significant evidence of positive causal impact after controls."

print("\nCausal conclusion:")
print(causal_conclusion)


# -------------------------
# Export final summary
# -------------------------
causal_summary.to_csv("final_causal_impact_summary.csv", index=False)

print("\nSaved file: final_causal_impact_summary.csv")

===== Final Causal Impact Summary Table =====


,Model,Rule_Coefficient,P_Value,CI_Lower,CI_Upper,R_Squared,Adj_R_Squared,Approx_Percentage_Effect_Log_Models,Interpretation
0,Model 1A - Baseline AOV,295.3146,0.0000,222.6911,367.9381,0.0033,0.0032,NaN,Positive and statistically significant
1,Model 2A - + BasketSize,26.3144,0.4200,-37.6436,90.2723,0.1972,0.1972,NaN,Not statistically significant
2,Model 3A - + BasketSize + AvgUnitPrice,28.1493,0.3878,-35.7302,92.0288,0.1975,0.1974,NaN,Not statistically significant
3,Model 4A - + BasketSize + AvgUnitPrice + Total...,-19.5486,0.3428,-59.9343,20.8371,0.7220,0.7220,NaN,Not statistically significant
4,Model 5A - + BasketSize + AvgUnitPrice + Total...,-20.1419,0.3277,-60.4795,20.1957,0.7250,0.7247,NaN,Not statistically significant
5,Model 1B - Baseline log_AOV,0.7166,0.0000,0.6320,0.8013,0.0035,0.0035,104.7503,Positive and statistically significant
6,Model 2B - log_AOV + BasketSize,0.0863,0.0429,0.0028,0.1698,0.1983,0.1983,9.0093,Positive and statistically significant
7,Model 3B - log_AOV + BasketSize + AvgUnitPrice,0.0858,0.0442,0.0022,0.1694,0.1984,0.1983,8.9599,Positive and statistically significant
8,Model 4B - log_AOV + BasketSize + AvgUnitPrice...,0.0085,0.8154,-0.0628,0.0797,0.4505,0.4504,0.8522,Not statistically significant
9,Model 5B - log_AOV + BasketSize + AvgUnitPrice...,-0.0035,0.9237,-0.0744,0.0674,0.4552,0.4545,-0.3459,Not statistically significant



===== Final Causal Interpretation =====
Final AOV model coefficient: -20.1419
Final AOV model p-value: 0.32774

Final log_AOV model coefficient: -0.003465
Final log_AOV percentage effect: -0.3459 %
Final log_AOV model p-value: 0.92369

Causal conclusion:
No statistically significant evidence of positive causal impact after controls.

Saved file: final_causal_impact_summary.csv
